# Workbook 8 — Revenue Driver Decomposition

## 1.1 Load Item-Level Sales Data (2025)

- Load item-level sales file for 2025.
- Inspect structure before transformation.
- Identify:
  - Date column
  - Revenue column
  - Quantity column
  - Order identifier (if available)

In [123]:
import pandas as pd
import matplotlib.pyplot as plt

In [124]:
item_2025 = pd.read_csv("../data/raw/item_sales_2025.csv")

ParserError: Error tokenizing data. C error: Expected 1 fields in line 9, saw 2


## 1.1a Fix Item-Level File Parsing Error (2025)

- ParserError indicates delimiter / metadata issues similar to Sales Reports.
- Use `engine="python"` and skip bad lines to load safely.
- Inspect first rows to confirm structure before cleaning.

In [125]:
item_2025 = pd.read_csv(
    "../data/raw/item_sales_2025.csv",
    sep=",",
    engine="python",
    on_bad_lines="skip"
)

item_2025.head()

,Items Report
0,"Jan 1, 2025 12:00 AM - Dec 31, 2025 11:59 PM"
1,"Requested on: Jan 12, 2026 6:30 PM"
2,Filters: Item Type = Revenue Items
3,Categories: All
4,"The report reflects all revenue items in paid,..."


## 1.2 Confirm True Header Row Index (Manual Validation)

- Manual inspection confirms true header begins on line 15.
- Python uses zero-based indexing.
- Therefore header_row_index = 14.
- We will hard-code this value for consistency.

In [126]:
header_row_index = 14

header_row_index

14

## 1.2a Lock `skiprows` for Item Report (2025)

- Lock header position using confirmed index.
- Remove metadata rows above true header.
- Reload dataset starting at correct header row.
- Confirm structure is clean.

In [127]:
skiprows_2025_items = 14

item_2025 = pd.read_csv(
    "../data/raw/item_sales_2025.csv",
    skiprows=skiprows_2025_items
)

item_2025.head()

,Category Name,Name,SKU,Product Code,Gross Sales,Net Sales,Sold,Refunded,Net Sold,Item Gross Sales,...,Discounts,Repayments,Refunds,Item Net Sales,Modifier Net Sales,% Net Sales,Avg Item Size,COGS,Gross Profit,Gross Profit Margin
0,Deals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Family Special (2 Lrg 2 Toppings),NaN,NaN,"$228,925.58","$228,671.21",7494,4,7490,"$224,075.58",...,-$129.39,$27.59,-$152.57,"$223,829.24","$4,841.97",18.51%,$30.53,$0.00,"$228,671.21",100%
2,NaN,NaN,,,,,,,,,...,,,,,,,,,,
3,NaN,NaN,,,,,,,,,...,,,,,,,,,,
4,NaN,NaN,,,,,,,,,...,,,,,,,,,,


## 1.2b Validate Column Names

- Confirm first column equals "Category Name".
- Ensure no metadata rows remain.

In [128]:
item_2025.columns

Index(['Category Name', 'Name', 'SKU', 'Product Code', 'Gross Sales',
       'Net Sales', 'Sold', 'Refunded', 'Net Sold', 'Item Gross Sales',
       'Modifier Name', 'Modifier Sold', 'Modifier Amount', 'Discounts',
       'Repayments', 'Refunds', 'Item Net Sales', 'Modifier Net Sales',
       '% Net Sales', 'Avg Item Size', 'COGS', 'Gross Profit',
       'Gross Profit Margin'],
      dtype='object')

## 1.3 Flatten Hierarchical Category Structure (2025)

- Clover export uses category rows as section headers.
- Category rows contain values only in "Category Name".
- Product rows contain sales metrics but Category Name may be NaN.
- We will forward-fill Category Name to propagate category labels

In [129]:
# Forward fill category names downward
item_2025["Category Name"] = item_2025["Category Name"].ffill()

item_2025.head(10)

,Category Name,Name,SKU,Product Code,Gross Sales,Net Sales,Sold,Refunded,Net Sold,Item Gross Sales,...,Discounts,Repayments,Refunds,Item Net Sales,Modifier Net Sales,% Net Sales,Avg Item Size,COGS,Gross Profit,Gross Profit Margin
0,Deals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Deals,Family Special (2 Lrg 2 Toppings),NaN,NaN,"$228,925.58","$228,671.21",7494,4,7490,"$224,075.58",...,-$129.39,$27.59,-$152.57,"$223,829.24","$4,841.97",18.51%,$30.53,$0.00,"$228,671.21",100%
2,Deals,NaN,,,,,,,,,...,,,,,,,,,,
3,Deals,NaN,,,,,,,,,...,,,,,,,,,,
4,Deals,NaN,,,,,,,,,...,,,,,,,,,,
5,Deals,NaN,,,,,,,,,...,,,,,,,,,,
6,Deals,NaN,,,,,,,,,...,,,,,,,,,,
7,Deals,NaN,,,,,,,,,...,,,,,,,,,,
8,Deals,NaN,,,,,,,,,...,,,,,,,,,,
9,Deals,NaN,,,,,,,,,...,,,,,,,,,,


## 1.3a Remove Pure Category Header Rows

- Category header rows contain no sales metrics.
- Remove rows where Gross Sales is NaN.
- Keep only rows with actual product data.

In [130]:
# Keep only rows with actual sales values
item_2025 = item_2025[item_2025["Gross Sales"].notna()].copy()

item_2025.head()

,Category Name,Name,SKU,Product Code,Gross Sales,Net Sales,Sold,Refunded,Net Sold,Item Gross Sales,...,Discounts,Repayments,Refunds,Item Net Sales,Modifier Net Sales,% Net Sales,Avg Item Size,COGS,Gross Profit,Gross Profit Margin
1,Deals,Family Special (2 Lrg 2 Toppings),NaN,NaN,"$228,925.58","$228,671.21",7494,4,7490,"$224,075.58",...,-$129.39,$27.59,-$152.57,"$223,829.24","$4,841.97",18.51%,$30.53,$0.00,"$228,671.21",100%
2,Deals,NaN,,,,,,,,,...,,,,,,,,,,
3,Deals,NaN,,,,,,,,,...,,,,,,,,,,
4,Deals,NaN,,,,,,,,,...,,,,,,,,,,
5,Deals,NaN,,,,,,,,,...,,,,,,,,,,


## 1.3b Remove Spacer Rows (No Product Name)

- Spacer rows contain Category Name but no product Name.
- Financial columns are NaN.
- These rows are formatting artifacts from Clover.
- Keep only rows where Name is not null.

In [131]:
# Keep only rows with valid product names
item_2025 = item_2025[item_2025["Name"].notna()].copy()

item_2025.head(10)

,Category Name,Name,SKU,Product Code,Gross Sales,Net Sales,Sold,Refunded,Net Sold,Item Gross Sales,...,Discounts,Repayments,Refunds,Item Net Sales,Modifier Net Sales,% Net Sales,Avg Item Size,COGS,Gross Profit,Gross Profit Margin
1,Deals,Family Special (2 Lrg 2 Toppings),NaN,NaN,"$228,925.58","$228,671.21",7494,4,7490,"$224,075.58",...,-$129.39,$27.59,-$152.57,"$223,829.24","$4,841.97",18.51%,$30.53,$0.00,"$228,671.21",100%
274,Deals,Three Seasons (XL),NaN,NaN,"$23,748.45","$23,743.58",968,0,968,"$23,586.25",...,-$4.87,$0.00,$0.00,"$23,581.38",$162.20,1.92%,$24.53,$0.00,"$23,743.58",100%
364,Deals,Large Pizza & Bread Sticks,,NaN,"$21,621.62","$21,607.20",995,0.00,995.00,"$20,577.77",...,-$14.42,$57.54,-$57.54,"$20,563.35","$1,043.85",1.75%,$21.72,$0.00,"$21,607.20",100%
449,Deals,2 Large Pizza One Topping,NaN,NaN,"$21,234.93","$21,214.72",783,0,783,"$21,116.43",...,-$20.21,$0.00,$0.00,"$21,096.22",$118.50,1.72%,$27.09,$0.00,"$21,214.72",100%
535,Deals,Pizza Party Pack (Medium Pizza and 6 Wings),NaN,NaN,"$8,035.27","$8,004.79",352,1,351,"$7,638.25",...,-$6.98,$0.00,-$23.50,"$7,610.72",$394.07,0.65%,$22.81,$0.00,"$8,004.79",100%
607,Deals,"Superbowl deal (XL, 10 wings, GB)",,,$110.34,$110.34,3,0,3,$108.00,...,$0.00,$0.00,$0.00,$108.00,$2.34,0.01%,$36.78,$0.00,$110.34,100%
617,Deals,3 Large Pizzas,,,$39.70,$39.70,1,0,1,$38.00,...,$0.00,$0.00,$0.00,$38.00,$1.70,0.00%,$39.70,$0.00,$39.70,100%
624,Pepperoni Pizza,Lrg Pepperoni Pizza,NaN,NaN,"$79,682.05","$79,413.93",4569,4,4565,"$75,807.54",...,-$198.88,$54.48,-$123.72,"$75,551.62","$3,862.31",6.43%,$17.40,$0.00,"$79,413.93",100%
678,Pepperoni Pizza,XL Pepperoni Pizza,NaN,NaN,"$46,876.54","$46,673.48",2239,0.96,2238.04,"$44,479.60",...,-$184.15,$39.44,-$58.34,"$44,278.34","$2,395.14",3.78%,$20.85,$0.00,"$46,673.48",100%
729,Pepperoni Pizza,Medium Pepperoni Pizza,NaN,NaN,"$31,367.50","$31,254.91",2250,1,2249,"$28,991.76",...,-$99.90,$0.00,-$12.69,"$28,881.36","$2,373.56",2.53%,$13.90,$0.00,"$31,254.91",100%


## 1.3c Validate Clean Structure

- Confirm no large vertical NaN gaps remain.
- Confirm each row represents one product.
- Confirm Category Name remains populated.

In [132]:
item_2025.info()

<class 'pandas.core.frame.DataFrame'>
Index: 567 entries, 1 to 4932
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Category Name        567 non-null    object
 1   Name                 567 non-null    object
 2   SKU                  449 non-null    object
 3   Product Code         437 non-null    object
 4   Gross Sales          567 non-null    object
 5   Net Sales            567 non-null    object
 6   Sold                 567 non-null    object
 7   Refunded             567 non-null    object
 8   Net Sold             567 non-null    object
 9   Item Gross Sales     567 non-null    object
 10  Modifier Name        567 non-null    object
 11  Modifier Sold        567 non-null    object
 12  Modifier Amount      567 non-null    object
 13  Discounts            567 non-null    object
 14  Repayments           567 non-null    object
 15  Refunds              567 non-null    object
 16  Item Net Sal

## 1.4 Normalize Data Types (2025 Item Report)

- Convert currency columns to numeric.
- Remove "$" and commas.
- Convert count columns to numeric.
- Convert margin percent to decimal float.
- Validate dtypes.

In [133]:
# --- Currency Columns ---
currency_cols = [
    "Gross Sales", "Net Sales", "Item Gross Sales",
    "Discounts", "Repayments", "Refunds",
    "Item Net Sales", "Modifier Net Sales",
    "Modifier Amount",
    "COGS", "Gross Profit"
]

for col in currency_cols:
    item_2025[col] = (
        item_2025[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    item_2025[col] = pd.to_numeric(item_2025[col], errors="coerce")

# --- Count Columns ---
count_cols = [
    "Sold", "Refunded", "Net Sold", "Modifier Sold"
]

for col in count_cols:
    item_2025[col] = pd.to_numeric(item_2025[col], errors="coerce")

# --- Percentage Columns ---
percent_cols = [
    "% Net Sales", "Gross Profit Margin"
]

for col in percent_cols:
    item_2025[col] = (
        item_2025[col]
        .astype(str)
        .str.replace("%", "", regex=False)
    )
    item_2025[col] = pd.to_numeric(item_2025[col], errors="coerce") / 100

item_2025.dtypes

Category Name           object
Name                    object
SKU                     object
Product Code            object
Gross Sales            float64
Net Sales              float64
Sold                     int64
Refunded               float64
Net Sold               float64
Item Gross Sales       float64
Modifier Name           object
Modifier Sold          float64
Modifier Amount        float64
Discounts              float64
Repayments             float64
Refunds                float64
Item Net Sales         float64
Modifier Net Sales     float64
% Net Sales            float64
Avg Item Size           object
COGS                   float64
Gross Profit           float64
Gross Profit Margin    float64
dtype: object

## 1.4.1 Standardize Numeric Columns (Counts + Avg Item Size)

- Convert count columns to numeric integers where possible.
- Preserve missing values using pandas nullable Int64 dtype.
- Clean Avg Item Size into a numeric column for later analysis.
- Keep original Avg Item Size column for traceability.

In [134]:
# --- Count columns: convert to nullable integers (keeps NaN) ---
count_cols = ["Sold", "Refunded", "Net Sold", "Modifier Sold"]

for col in count_cols:
    item_2025[col] = pd.to_numeric(item_2025[col], errors="coerce")
    item_2025[col] = item_2025[col].round(0).astype("Int64")

# --- Avg Item Size: create numeric version (keep original) ---
item_2025["Avg Item Size Num"] = (
    item_2025["Avg Item Size"]
    .astype(str)
    .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
)

item_2025["Avg Item Size Num"] = pd.to_numeric(
    item_2025["Avg Item Size Num"],
    errors="coerce"
)

item_2025[count_cols + ["Avg Item Size", "Avg Item Size Num"]].head(10)

,Sold,Refunded,Net Sold,Modifier Sold,Avg Item Size,Avg Item Size Num
1,7494,4,7490,<NA>,$30.53,30.53
274,968,0,968,<NA>,$24.53,24.53
364,995,0,995,<NA>,$21.72,21.72
449,783,0,783,<NA>,$27.09,27.09
535,352,1,351,<NA>,$22.81,22.81
607,3,0,3,<NA>,$36.78,36.78
617,1,0,1,<NA>,$39.70,39.70
624,4569,4,4565,<NA>,$17.40,17.40
678,2239,1,2238,<NA>,$20.85,20.85
729,2250,1,2249,<NA>,$13.90,13.90


## 1.4.2 Validate Dtypes After Standardization

- Confirm counts are Int64 (nullable integer).
- Confirm Avg Item Size Num is numeric.
- Confirm financial columns remain float64.

In [135]:
item_2025.dtypes

Category Name           object
Name                    object
SKU                     object
Product Code            object
Gross Sales            float64
Net Sales              float64
Sold                     Int64
Refunded                 Int64
Net Sold                 Int64
Item Gross Sales       float64
Modifier Name           object
Modifier Sold            Int64
Modifier Amount        float64
Discounts              float64
Repayments             float64
Refunds                float64
Item Net Sales         float64
Modifier Net Sales     float64
% Net Sales            float64
Avg Item Size           object
COGS                   float64
Gross Profit           float64
Gross Profit Margin    float64
Avg Item Size Num      float64
dtype: object

## 1.5 Create Base vs Modifier Revenue Components

- Separate base item revenue from modifier revenue.
- Base revenue = Item Net Sales.
- Modifier revenue = Modifier Net Sales.
- Total realized revenue = Base + Modifier.
- Validate reconciliation against Net Sales.

In [136]:
# --- Create Revenue Components ---
item_2025["Base Revenue"] = item_2025["Item Net Sales"]

item_2025["Modifier Revenue"] = item_2025["Modifier Net Sales"]

item_2025["Total Revenue"] = (
    item_2025["Base Revenue"] +
    item_2025["Modifier Revenue"]
)

# --- Validate reconciliation ---
item_2025[[
    "Net Sales",
    "Base Revenue",
    "Modifier Revenue",
    "Total Revenue"
]].head(10)

,Net Sales,Base Revenue,Modifier Revenue,Total Revenue
1,228671.21,223829.24,4841.97,228671.21
274,23743.58,23581.38,162.20,23743.58
364,21607.20,20563.35,1043.85,21607.20
449,21214.72,21096.22,118.50,21214.72
535,8004.79,7610.72,394.07,8004.79
607,110.34,108.00,2.34,110.34
617,39.70,38.00,1.70,39.70
624,79413.93,75551.62,3862.31,79413.93
678,46673.48,44278.34,2395.14,46673.48
729,31254.91,28881.36,2373.56,31254.92


## 1.5.1 Validate Revenue Reconciliation

- Compare Net Sales vs Total Revenue.
- Small rounding differences are acceptable.
- Large gaps indicate structural issue.

In [137]:
item_2025["Revenue Diff"] = (
    item_2025["Net Sales"] -
    item_2025["Total Revenue"]
)

item_2025["Revenue Diff"].describe()

count    567.000000
mean       0.000141
std        0.001322
min       -0.010000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.010000
Name: Revenue Diff, dtype: float64

## 1.5.2 Investigate Non-Zero Revenue Differences

- Identify rows where Revenue Diff ≠ 0.
- Determine whether differences are rounding-related.
- Confirm no systemic mismatch between Net Sales and components.

In [138]:
# Filter non-zero differences
diff_check = item_2025[item_2025["Revenue Diff"].abs() > 0.01]

diff_check[[
    "Category Name",
    "Name",
    "Net Sales",
    "Total Revenue",
    "Revenue Diff"
]].head(10)

diff_check["Revenue Diff"].describe()

count    6.000000
mean     0.006667
std      0.008165
min     -0.010000
25%      0.010000
50%      0.010000
75%      0.010000
max      0.010000
Name: Revenue Diff, dtype: float64

## 1.5.3 Confirm Revenue Integrity

- Revenue Diff count > 0 is minimal.
- Differences are likely rounding artifacts.
- No structural mismatch detected.
- Revenue decomposition can proceed.

In [139]:
# Count of material differences
(item_2025["Revenue Diff"].abs() > 0.01).sum()

6

## 1.6 Calculate Implied Price per Unit (Base Only)

- Implied Base Price = Base Revenue / Net Sold.
- Avoid divide-by-zero errors.
- This represents realized average selling price.

In [140]:
item_2025["Implied Base Price"] = (
    item_2025["Base Revenue"] /
    item_2025["Net Sold"].replace(0, pd.NA)
)

item_2025[[
    "Name",
    "Net Sold",
    "Base Revenue",
    "Implied Base Price"
]].head(10)

,Name,Net Sold,Base Revenue,Implied Base Price
1,Family Special (2 Lrg 2 Toppings),7490,223829.24,29.883744
274,Three Seasons (XL),968,23581.38,24.36093
364,Large Pizza & Bread Sticks,995,20563.35,20.666683
449,2 Large Pizza One Topping,783,21096.22,26.94281
535,Pizza Party Pack (Medium Pizza and 6 Wings),351,7610.72,21.682963
607,"Superbowl deal (XL, 10 wings, GB)",3,108.00,36.0
617,3 Large Pizzas,1,38.00,38.0
624,Lrg Pepperoni Pizza,4565,75551.62,16.550191
678,XL Pepperoni Pizza,2238,44278.34,19.784781
729,Medium Pepperoni Pizza,2249,28881.36,12.841867


## 1.6.1 Standardize Menu Pricing File Naming

- Rename uploaded file to `menu_pricing_2025.xlsx`.
- Store in raw data folder for consistency.
- Update notebook path to match standardized naming.

In [141]:
menu_path = "../data/raw/menu_pricing_2025.xlsx"
menu_path

'../data/raw/menu_pricing_2025.xlsx'

## 1.6.2 Load Menu Pricing Workbook

- Load Excel file.
- Inspect available sheet names.
- Identify which sheet(s) contain base item pricing.
- Do not merge yet.

In [142]:
menu_file = pd.ExcelFile(menu_path)

menu_file.sheet_names

/Users/user/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Instructions & Glossary',
 'Items',
 'Modifier Groups',
 'Categories',
 'Tax Rates']

## 1.6.3 Inspect Each Menu Sheet Structure

- Load each sheet individually.
- Preview first 10 rows.
- Identify:
  - Product name column
  - SKU or Product Code column
  - Base price column
- Determine correct join key.

In [143]:
# Load all sheets into dictionary for inspection
menu_sheets = {
    sheet: pd.read_excel(menu_path, sheet_name=sheet)
    for sheet in menu_file.sheet_names
}

# Preview first sheet (change index if needed)
first_sheet_name = menu_file.sheet_names[0]

menu_sheets[first_sheet_name].head(10)

/Users/user/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/user/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/user/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/user/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Clover Inventory Import/Export,Unnamed: 1
0,Instructions: Complete each tab of the sheet w...,NaN
1,Glossary,NaN
2,Term,Description
3,Alternative item name,An alternative item name can be used for inter...
4,Categories,A group of similar items.
5,Category name,Set category names to group similar items in y...
6,Clover ID,Leave blank.
7,Default Tax Rates,Would you like to apply a default tax rate? If...
8,Description,"A brief, customer-friendly description of your..."
9,Hidden,This is used to indicate if an item is shown o...


## 1.6.4 Load Menu Items Sheet (Pricing Source)

- Use the sheet that contains the Price column ("Items").
- Load only this sheet for pricing integration.
- Preview first 10 rows to confirm structure.

In [144]:
menu_items = pd.read_excel(
    menu_path,
    sheet_name="Items"
)

menu_items.head(50)

/Users/user/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Clover ID,Name,Alternate Name,Description,Price,Price Type,Price Unit,Cost,Product Code,SKU,Quantity,Hidden?,Default tax rates?,Non-revenue item?,Printer Labels,Modifier Groups,Categories,Tax Rates,Variant Attribute,Variant Option
0,YNQ0KN914BRET,Combination Pizza,NaN,NaN,12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,No,No,Specialty Pizzas,NaN,NaN,Sales Tax,NaN,NaN
1,B4Z000QGRVXR0,Garlic Chicken Pizza,NaN,NaN,12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,Yes,No,Sides,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Specialty Pizzas,NaN,NaN,NaN,NaN,NaN
3,1T4A7EAV3BRPY,All Meat Pizza,NaN,"Red tomato sauce, mozzarella cheese, Canadian ...",12.88,Fixed,NaN,0.0,NaN,NaN,NaN,Yes,No,No,NaN,large all meat,Specialties,Sales Tax,NaN,NaN
4,6H4EPSWHBKFTG,Bacon Chicken Pizza,NaN,"White garlic sauce, mozzarella cheese, red oni...",12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,No,No,Specialty Pizzas,NaN,NaN,Sales Tax,NaN,NaN
5,MWG5PXJ39CTJ6,BBQ Chicken Pizza,NaN,NaN,12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,No,No,Specialty Pizzas,NaN,NaN,Sales Tax,NaN,NaN
6,538Z2KGVST7C0,Chef's Special Pizza,NaN,NaN,12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,No,No,Specialty Pizzas,NaN,NaN,Sales Tax,NaN,NaN
7,S946YPAR16MXT,Garlic Delight Pizza,NaN,NaN,12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,Yes,No,Specialty Pizzas,NaN,NaN,NaN,NaN,NaN
8,BP5M73X331G70,Greek Pizza,NaN,NaN,12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,Yes,No,Specialty Pizzas,NaN,NaN,NaN,NaN,NaN
9,N5V49YTF277S6,Hawaiian Delight Pizza,NaN,NaN,12.88,Fixed,NaN,0.0,NaN,NaN,NaN,No,Yes,No,Specialty Pizzas,NaN,NaN,NaN,NaN,NaN


## 1.6.5 Select Required Menu Columns

- Keep only columns needed for merge and validation:
  - Name
  - Price
  - SKU
  - Product Code
- Rename Price to Menu Price for clarity.

In [145]:
menu_items = menu_items[[
    "Name",
    "Price",
    "SKU",
    "Product Code"
]].copy()

menu_items = menu_items.rename(columns={
    "Price": "Menu Price"
})

menu_items.head(50)

,Name,Menu Price,SKU,Product Code
0,Combination Pizza,12.88,NaN,NaN
1,Garlic Chicken Pizza,12.88,NaN,NaN
2,NaN,NaN,NaN,NaN
3,All Meat Pizza,12.88,NaN,NaN
4,Bacon Chicken Pizza,12.88,NaN,NaN
5,BBQ Chicken Pizza,12.88,NaN,NaN
6,Chef's Special Pizza,12.88,NaN,NaN
7,Garlic Delight Pizza,12.88,NaN,NaN
8,Greek Pizza,12.88,NaN,NaN
9,Hawaiian Delight Pizza,12.88,NaN,NaN


## 1.6.6 Confirm Variant Pricing Is Encoded in Name

- Size variants appear as separate item names (e.g., Med Greek Pizza, Lrg Greek Pizza).
- This means each size has its own Menu Price.
- No separate variant table is required.
- Safe to merge directly on Name.

In [146]:
# Sanity check: confirm multiple size variants exist as separate rows
menu_items[menu_items["Name"].str.contains("Garlic Chicken Pizza", case=False, na=False)][
    ["Name", "Menu Price"]
]

,Name,Menu Price
1,Garlic Chicken Pizza,12.88
388,Lrg Garlic Chicken Pizza,24.60
399,XL Garlic Chicken Pizza,27.08
410,SL Garlic Chicken Pizza,34.48
421,Med Garlic Chicken Pizza,19.49
794,Small Garlic Chicken Pizza,13.84
1176,Popmenu Med Garlic Chicken Pizza,19.49
1182,Popmenu Lrg Garlic Chicken Pizza,24.60
1188,Popmenu SL Garlic Chicken Pizza,34.48
1194,Popmenu XL Garlic Chicken Pizza,27.08


### 1.6.6a Validate Menu Name Uniqueness

- Objective:
  - Confirm menu.Name is unique before merge.
- Acceptance Criteria:
  - Zero duplicates.
  - Duplicate list must be empty.

### 1.6.6b Diagnose Duplicate Source

- Objective:
  - Confirm that menu.Name contains systematic duplicates.
- Expected Finding:
  - Each item appears exactly twice.

In [147]:
menu_name_dupes = (
    menu_df["Name"]
    .value_counts()
    .loc[lambda s: s > 1]
)

menu_name_dupes

NameError: name 'menu_df' is not defined

### 1.6.6c Deduplicate Menu Table

- Objective:
  - Remove duplicate Name rows before merge.
- Assumption:
  - Duplicate rows are structurally identical.
- Action:
  - Keep first occurrence per Name.

In [148]:
menu_items = (
    menu_items
    .drop_duplicates(subset=["Name"])
    .copy()
)

menu_items.shape

(225, 4)

### 1.6.6d Re-validate Name Uniqueness

- Objective:
  - Confirm zero duplicates remain.
- Acceptance Criteria:
  - Returned Series must be empty.

In [149]:
menu_items["Name"].value_counts().loc[lambda s: s > 1]

Series([], Name: count, dtype: int64)

### 1.6.6e Merge Item Sales to Menu

- Objective:
  - Attach Menu Price to transactional item-level data.
- Join Structure:
  - Left join from item_sales → menu_items.
  - Enforce validate="m:1".
- Expected Result:
  - Each transaction row maps to exactly one Menu Price.

In [150]:
item_2025_merged = (
    item_2025
    .merge(
        menu_items,
        how="left",
        left_on="Name",
        right_on="Name",
        validate="m:1"
    )
    .copy()
)

item_2025_merged.head()

,Category Name,Name,SKU_x,Product Code_x,Gross Sales,Net Sales,Sold,Refunded,Net Sold,Item Gross Sales,...,Gross Profit Margin,Avg Item Size Num,Base Revenue,Modifier Revenue,Total Revenue,Revenue Diff,Implied Base Price,Menu Price,SKU_y,Product Code_y
0,Deals,Family Special (2 Lrg 2 Toppings),NaN,NaN,228925.58,228671.21,7494,4,7490,224075.58,...,1.0,30.53,223829.24,4841.97,228671.21,0.000000e+00,29.883744,30.85,NaN,NaN
1,Deals,Three Seasons (XL),NaN,NaN,23748.45,23743.58,968,0,968,23586.25,...,1.0,24.53,23581.38,162.20,23743.58,0.000000e+00,24.36093,24.37,NaN,NaN
2,Deals,Large Pizza & Bread Sticks,,NaN,21621.62,21607.20,995,0,995,20577.77,...,1.0,21.72,20563.35,1043.85,21607.20,3.637979e-12,20.666683,21.24,NaN,NaN
3,Deals,2 Large Pizza One Topping,NaN,NaN,21234.93,21214.72,783,0,783,21116.43,...,1.0,27.09,21096.22,118.50,21214.72,0.000000e+00,26.94281,28.23,NaN,NaN
4,Deals,Pizza Party Pack (Medium Pizza and 6 Wings),NaN,NaN,8035.27,8004.79,352,1,351,7638.25,...,1.0,22.81,7610.72,394.07,8004.79,0.000000e+00,21.682963,22.67,NaN,NaN


### 1.6.6f Detect Unmatched Item Names

- Objective:
  - Identify transaction item names not found in menu_items after merge.
- Required Outputs:
  - Unique unmatched item count.
  - Share of total transaction rows unmatched.
- Acceptance Criteria:
  - Unmatched share must be small and explainable before proceeding to pricing analysis.

In [151]:
unmatched = item_2025_merged[
    item_2025_merged["Menu Price"].isna()
].copy()

unmatched_unique = unmatched["Name"].nunique()

unmatched_share = (
    unmatched.shape[0] /
    item_2025_merged.shape[0]
)

unmatched_unique, unmatched_share

(46, 0.7601410934744268)

### 1.6.6g Summarize Unmatched Revenue Exposure

- Objective:
  - Quantify business exposure of unmatched items.
- Required Outputs:
  - Row count
  - Quantity
  - Total Revenue
  - Ranked by revenue contribution
- Interpretation:
  - High-revenue unmatched items must be resolved before pricing validation.

In [152]:
unmatched_summary = (
    unmatched
    .groupby("Name")
    .agg(
        rows=("Name", "size"),
        quantity=("Sold", "sum"),
        revenue=("Total Revenue", "sum")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

unmatched_summary.head(20)

,Name,rows,quantity,revenue
4,Medium Pepperoni Pizza,1,2250,31254.92
2,Manual Transaction,323,1531,18681.00
3,Medium Cheese Pizza,1,432,5005.79
5,S-Large Half and Half,1,90,2546.82
1,Custom Item,58,400,1828.60
6,Small Garlic Lover Pizza,1,80,1119.47
0,Cancels,1,1,26.00
45,water,2,4,10.00
10,cancelled pizza,1,1,9.00
9,canceled order,1,1,7.00


## 1.6.7 Validate Menu Price vs Implied Base Price

- Objective:
  - Compare Menu Price to Implied Base Price.
- Formula:
  - Implied Price = Item Net Sales / Sold
  - Delta = Implied Price − Menu Price
- Output:
  - Absolute delta
  - Percentage delta

In [153]:
item_2025_merged["Implied Price Calc"] = (
    item_2025_merged["Item Net Sales"] /
    item_2025_merged["Sold"]
)

item_2025_merged["Price Delta"] = (
    item_2025_merged["Implied Price Calc"] -
    item_2025_merged["Menu Price"]
)

item_2025_merged["Price Delta Pct"] = (
    item_2025_merged["Price Delta"] /
    item_2025_merged["Menu Price"]
)

### 1.6.7a Summarize Price Deviations by Item

- Objective:
  - Aggregate pricing deviations at item level.
- Output:
  - Avg implied price
  - Avg menu price
  - Avg delta
  - Avg delta %

In [154]:
price_deviation_summary = (
    item_2025_merged
    .groupby("Name")
    .agg(
        total_net_sales=("Item Net Sales", "sum"),
        total_sold=("Sold", "sum"),
        avg_menu_price=("Menu Price", "mean"),
        total_revenue=("Total Revenue", "sum")
    )
    .reset_index()
)

price_deviation_summary["weighted_implied_price"] = (
    price_deviation_summary["total_net_sales"] /
    price_deviation_summary["total_sold"]
)

price_deviation_summary["delta"] = (
    price_deviation_summary["weighted_implied_price"] -
    price_deviation_summary["avg_menu_price"]
)

price_deviation_summary["delta_pct"] = (
    price_deviation_summary["delta"] /
    price_deviation_summary["avg_menu_price"]
)

price_deviation_summary = (
    price_deviation_summary
    .sort_values("delta_pct", ascending=False)
)

price_deviation_summary.head(20)

,Name,total_net_sales,total_sold,avg_menu_price,total_revenue,weighted_implied_price,delta,delta_pct
25,Garlic,108.59,199,0.46,108.59,0.545678,0.085678,0.186257
10,Buffalo,70.15,137,0.46,70.15,0.512044,0.052044,0.113139
50,Marinara,337.91,680,0.46,337.91,0.496926,0.036926,0.080275
97,Small Cheese Pizza,3383.04,394,8.05,3426.84,8.586396,0.536396,0.066633
60,Med Half and Half,5223.92,464,10.58,6799.24,11.258448,0.678448,0.064126
7,BBQ,87.91,181,0.46,106.41,0.485691,0.025691,0.055849
139,XS Cheese Pizza,2845.39,407,6.67,2875.40,6.99113,0.32113,0.048145
73,Ranch,10387.33,21931,0.46,10387.33,0.473637,0.013637,0.029645
24,Feta Bread,192.73,27,6.99,192.73,7.138148,0.148148,0.021194
5,Anchovies,64.21,23,2.75,64.21,2.791739,0.041739,0.015178


### 1.6.7b Flag Material Pricing Deviations

- Objective:
  - Identify items with meaningful pricing deviation.
- Criteria:
  - Absolute delta_pct > 0.05
  - total_sold > 100
- Output:
  - Filtered list of material deviations
  - Sorted by delta_pct (descending)

In [155]:
material_price_flags = (
    price_deviation_summary
    .loc[
        (price_deviation_summary["delta_pct"].abs() > 0.05) &
        (price_deviation_summary["total_sold"] > 100)
    ]
    .sort_values("delta_pct", ascending=False)
)

material_price_flags

,Name,total_net_sales,total_sold,avg_menu_price,total_revenue,weighted_implied_price,delta,delta_pct
25,Garlic,108.59,199,0.46,108.59,0.545678,0.085678,0.186257
10,Buffalo,70.15,137,0.46,70.15,0.512044,0.052044,0.113139
50,Marinara,337.91,680,0.46,337.91,0.496926,0.036926,0.080275
97,Small Cheese Pizza,3383.04,394,8.05,3426.84,8.586396,0.536396,0.066633
60,Med Half and Half,5223.92,464,10.58,6799.24,11.258448,0.678448,0.064126
7,BBQ,87.91,181,0.46,106.41,0.485691,0.025691,0.055849
37,Lrg Garlic Chicken Pizza,12458.10,534,24.60,12629.92,23.329775,-1.270225,-0.051635
118,Wings (6pc),9715.08,1072,9.56,9744.52,9.062575,-0.497425,-0.052032
19,Chicken Strips (10 pcs),3835.19,558,7.26,3835.19,6.8731,-0.3869,-0.053292
127,XL Garlic Chicken Pizza,12097.95,473,27.08,12192.24,25.577061,-1.502939,-0.0555


### 1.6.7c Recompute Realized Price Including Modifiers

- Objective:
  - Recompute realized price using full transactional revenue (base + modifiers).
- Definitions:
  - Realized Revenue = Total Revenue
  - Realized Price = Total Revenue / Sold
- Output:
  - Realized Price (row-level)
  - Realized Price Delta vs Menu Price
  - Realized Price Delta % vs Menu Price

In [156]:
item_2025_merged["Realized Price"] = (
    item_2025_merged["Total Revenue"] /
    item_2025_merged["Sold"]
)

item_2025_merged["Realized Price Delta"] = (
    item_2025_merged["Realized Price"] -
    item_2025_merged["Menu Price"]
)

item_2025_merged["Realized Price Delta Pct"] = (
    item_2025_merged["Realized Price Delta"] /
    item_2025_merged["Menu Price"]
)

### 1.6.7d Summarize Realized Price Deviations by Item (Weighted)

- Objective:
  - Compute weighted realized price vs menu price at item level.
- Formulas:
  - Weighted Realized Price = sum(Total Revenue) / sum(Sold)
  - Delta = Weighted Realized Price − avg(Menu Price)
  - Delta % = Delta / avg(Menu Price)
- Output:
  - Weighted realized price
  - Menu price
  - Delta
  - Delta %
  - Volume context (total_sold)

In [157]:
realized_price_summary = (
    item_2025_merged
    .groupby("Name")
    .agg(
        total_revenue=("Total Revenue", "sum"),
        total_sold=("Sold", "sum"),
        avg_menu_price=("Menu Price", "mean")
    )
    .reset_index()
)

realized_price_summary["weighted_realized_price"] = (
    realized_price_summary["total_revenue"] /
    realized_price_summary["total_sold"]
)

realized_price_summary["delta"] = (
    realized_price_summary["weighted_realized_price"] -
    realized_price_summary["avg_menu_price"]
)

realized_price_summary["delta_pct"] = (
    realized_price_summary["delta"] /
    realized_price_summary["avg_menu_price"]
)

realized_price_summary = (
    realized_price_summary
    .sort_values("delta_pct", ascending=False)
)

realized_price_summary.head(20)

,Name,total_revenue,total_sold,avg_menu_price,weighted_realized_price,delta,delta_pct
60,Med Half and Half,6799.24,464,10.58,14.653534,4.073534,0.385022
91,SL Multi Topping,3992.84,138,20.92,28.933623,8.013623,0.38306
67,Medium Multi Topping,18006.48,1169,11.49,15.403319,3.913319,0.340585
136,XL Multi Topping,14790.05,622,17.93,23.778215,5.848215,0.326169
108,Small Multi Topping,9392.08,810,8.78,11.59516,2.81516,0.320633
113,Square Multi Topping,3264.93,136,18.71,24.006838,5.296838,0.283102
46,Lrg Multi Topping,31023.31,1590,15.22,19.511516,4.291516,0.281966
7,BBQ,106.41,181,0.46,0.587901,0.127901,0.278045
112,Square Half and Half,1952.34,86,18.86,22.701628,3.841628,0.203692
141,XS Pizza,25546.83,2986,7.17,8.555536,1.385536,0.193241


### 1.6.7e Re-flag Material Pricing Deviations (Realized Price)

- Objective:
  - Flag material deviations using realized price (base + modifiers).
- Criteria:
  - Absolute delta_pct > 0.05
  - total_sold > 100
- Output:
  - Material deviation list based on realized pricing truth

In [158]:
material_realized_flags = (
    realized_price_summary
    .loc[
        (realized_price_summary["delta_pct"].abs() > 0.05) &
        (realized_price_summary["total_sold"] > 100)
    ]
    .sort_values("delta_pct", ascending=False)
)

material_realized_flags

,Name,total_revenue,total_sold,avg_menu_price,weighted_realized_price,delta,delta_pct
60,Med Half and Half,6799.24,464,10.58,14.653534,4.073534,0.385022
91,SL Multi Topping,3992.84,138,20.92,28.933623,8.013623,0.38306
67,Medium Multi Topping,18006.48,1169,11.49,15.403319,3.913319,0.340585
136,XL Multi Topping,14790.05,622,17.93,23.778215,5.848215,0.326169
108,Small Multi Topping,9392.08,810,8.78,11.59516,2.81516,0.320633
113,Square Multi Topping,3264.93,136,18.71,24.006838,5.296838,0.283102
46,Lrg Multi Topping,31023.31,1590,15.22,19.511516,4.291516,0.281966
7,BBQ,106.41,181,0.46,0.587901,0.127901,0.278045
141,XS Pizza,25546.83,2986,7.17,8.555536,1.385536,0.193241
25,Garlic,108.59,199,0.46,0.545678,0.085678,0.186257


In [159]:
item_2025.head()

,Category Name,Name,SKU,Product Code,Gross Sales,Net Sales,Sold,Refunded,Net Sold,Item Gross Sales,...,Avg Item Size,COGS,Gross Profit,Gross Profit Margin,Avg Item Size Num,Base Revenue,Modifier Revenue,Total Revenue,Revenue Diff,Implied Base Price
1,Deals,Family Special (2 Lrg 2 Toppings),NaN,NaN,228925.58,228671.21,7494,4,7490,224075.58,...,$30.53,0.0,228671.21,1.0,30.53,223829.24,4841.97,228671.21,0.000000e+00,29.883744
274,Deals,Three Seasons (XL),NaN,NaN,23748.45,23743.58,968,0,968,23586.25,...,$24.53,0.0,23743.58,1.0,24.53,23581.38,162.20,23743.58,0.000000e+00,24.36093
364,Deals,Large Pizza & Bread Sticks,,NaN,21621.62,21607.20,995,0,995,20577.77,...,$21.72,0.0,21607.20,1.0,21.72,20563.35,1043.85,21607.20,3.637979e-12,20.666683
449,Deals,2 Large Pizza One Topping,NaN,NaN,21234.93,21214.72,783,0,783,21116.43,...,$27.09,0.0,21214.72,1.0,27.09,21096.22,118.50,21214.72,0.000000e+00,26.94281
535,Deals,Pizza Party Pack (Medium Pizza and 6 Wings),NaN,NaN,8035.27,8004.79,352,1,351,7638.25,...,$22.81,0.0,8004.79,1.0,22.81,7610.72,394.07,8004.79,0.000000e+00,21.682963


## 1.6.8 Discount Penetration Analysis

- Objective:
  - Quantify how much realized price compression is explained by discounts.
- Definitions:
  - Discount Amount = Discounts
  - Discount Rate = Discounts / Total Revenue
  - Net Revenue = Total Revenue - Discounts
- Outputs:
  - Discount penetration by item
  - Discount rate distribution
  - Recomputed realized price using Net Revenue

In [160]:
required_cols = [
    "Name",
    "Sold",
    "Total Revenue",
    "Discounts"
]

missing_cols = [c for c in required_cols if c not in item_2025_merged.columns]
missing_cols

[]

### 1.6.8a Clean Discount Column

- Objective:
  - Ensure Discounts is numeric and non-null.
- Rules:
  - Coerce to numeric
  - Fill missing with 0

In [161]:
item_2025_merged["Discounts"] = (
    pd.to_numeric(item_2025_merged["Discounts"], errors="coerce")
    .fillna(0)
)

### 1.6.8b Compute Row-Level Discount Rate

- Objective:
  - Compute discount rate per row.
- Formula:
  - Discount Rate = Discounts / Total Revenue
- Guardrail:
  - Avoid divide-by-zero

In [162]:
denom = item_2025_merged["Total Revenue"].replace(0, pd.NA)

item_2025_merged["Discount Rate"] = (
    item_2025_merged["Discounts"] / denom
).fillna(0)

### 1.6.8c Aggregate Discount Penetration by Item

- Objective:
  - Quantify discount penetration for each item.
- Outputs:
  - total_discounts
  - total_revenue
  - discount_rate (weighted)
  - total_sold
- Formula:
  - Weighted Discount Rate = sum(Discounts) / sum(Total Revenue)

In [163]:
discount_summary = (
    item_2025_merged
    .groupby("Name")
    .agg(
        total_revenue=("Total Revenue", "sum"),
        total_discounts=("Discounts", "sum"),
        total_sold=("Sold", "sum"),
        avg_menu_price=("Menu Price", "mean"),
    )
    .reset_index()
)

discount_summary["weighted_discount_rate"] = (
    discount_summary["total_discounts"] /
    discount_summary["total_revenue"].replace(0, pd.NA)
).fillna(0)

discount_summary = discount_summary.sort_values(
    "weighted_discount_rate",
    ascending=False
)

discount_summary.head(20)

,Name,total_revenue,total_discounts,total_sold,avg_menu_price,weighted_discount_rate
90,SL Mexican Taco Pizza,301.23,0.0,9,34.48,0.0
148,garlic sauce side,5.00,0.0,10,NaN,0.0
146,feta,0.50,0.0,1,NaN,0.0
145,custom,3.00,0.0,3,1.00,0.0
144,cancelled pizza,9.00,0.0,1,NaN,0.0
143,canceled order,7.00,0.0,1,NaN,0.0
142,bell pepper on the side,0.50,0.0,1,NaN,0.0
134,XL House Specialty Pizza,988.73,0.0,36,27.08,0.0
128,XL Garlic Delight Pizza,498.98,0.0,18,27.08,0.0
120,Worker Sergei,7.00,0.0,1,NaN,0.0


### 1.6.8d Recompute Net Realized Price vs Menu Price

- Objective:
  - Compute realized price using net revenue after discounts.
- Formulas:
  - Net Revenue = Total Revenue - Discounts
  - Net Realized Price = sum(Net Revenue) / sum(Sold)
  - Delta % = (Net Realized Price - avg(Menu Price)) / avg(Menu Price)
- Output:
  - Net realized pricing deviation

In [164]:
net_price_summary = (
    item_2025_merged
    .assign(net_revenue=lambda df: df["Total Revenue"] - df["Discounts"])
    .groupby("Name")
    .agg(
        total_net_revenue=("net_revenue", "sum"),
        total_sold=("Sold", "sum"),
        avg_menu_price=("Menu Price", "mean")
    )
    .reset_index()
)

net_price_summary["net_realized_price"] = (
    net_price_summary["total_net_revenue"] /
    net_price_summary["total_sold"].replace(0, pd.NA)
)

net_price_summary["delta"] = (
    net_price_summary["net_realized_price"] -
    net_price_summary["avg_menu_price"]
)

net_price_summary["delta_pct"] = (
    net_price_summary["delta"] /
    net_price_summary["avg_menu_price"].replace(0, pd.NA)
)

net_price_summary = net_price_summary.sort_values(
    "delta_pct",
    ascending=False
)

net_price_summary.head(20)

,Name,total_net_revenue,total_sold,avg_menu_price,net_realized_price,delta,delta_pct
60,Med Half and Half,6805.86,464,10.58,14.667802,4.087802,0.386371
91,SL Multi Topping,3992.84,138,20.92,28.933623,8.013623,0.38306
67,Medium Multi Topping,18156.68,1169,11.49,15.531805,4.041805,0.351767
136,XL Multi Topping,14890.19,622,17.93,23.939212,6.009212,0.335148
108,Small Multi Topping,9456.15,810,8.78,11.674259,2.894259,0.329642
46,Lrg Multi Topping,31150.88,1590,15.22,19.591748,4.371748,0.287237
113,Square Multi Topping,3270.07,136,18.71,24.044632,5.334632,0.285122
7,BBQ,106.41,181,0.46,0.587901,0.127901,0.278045
112,Square Half and Half,1952.34,86,18.86,22.701628,3.841628,0.203692
141,XS Pizza,25638.65,2986,7.17,8.586286,1.416286,0.197529


### 1.6.8e Flag Material Net Pricing Deviations

- Objective:
  - Flag items with material net realized pricing deviation.
- Criteria:
  - Absolute delta_pct > 0.05
  - total_sold > 100

In [165]:
material_net_flags = (
    net_price_summary
    .loc[
        (net_price_summary["delta_pct"].abs() > 0.05) &
        (net_price_summary["total_sold"] > 100)
    ]
    .sort_values("delta_pct", ascending=False)
)

material_net_flags

,Name,total_net_revenue,total_sold,avg_menu_price,net_realized_price,delta,delta_pct
60,Med Half and Half,6805.86,464,10.58,14.667802,4.087802,0.386371
91,SL Multi Topping,3992.84,138,20.92,28.933623,8.013623,0.38306
67,Medium Multi Topping,18156.68,1169,11.49,15.531805,4.041805,0.351767
136,XL Multi Topping,14890.19,622,17.93,23.939212,6.009212,0.335148
108,Small Multi Topping,9456.15,810,8.78,11.674259,2.894259,0.329642
46,Lrg Multi Topping,31150.88,1590,15.22,19.591748,4.371748,0.287237
113,Square Multi Topping,3270.07,136,18.71,24.044632,5.334632,0.285122
7,BBQ,106.41,181,0.46,0.587901,0.127901,0.278045
141,XS Pizza,25638.65,2986,7.17,8.586286,1.416286,0.197529
25,Garlic,108.59,199,0.46,0.545678,0.085678,0.186257


## 1.7 Revenue Driver Decomposition (Volume vs Price)

- Objective:
  - Decompose revenue into volume and price components.
- Identity:
  - Revenue = Quantity × Realized Price
- Definitions:
  - Quantity = Sold
  - Realized Price = Net Realized Price (after discounts)
- Outputs:
  - Volume effect
  - Price effect
  - Contribution share

### 1.7a Aggregate Item-Level Totals (Net Revenue Basis)

In [166]:
driver_base = (
    item_2025_merged
    .assign(net_revenue=lambda df: df["Total Revenue"] - df["Discounts"])
    .groupby("Name")
    .agg(
        total_sold=("Sold", "sum"),
        total_net_revenue=("net_revenue", "sum"),
        avg_menu_price=("Menu Price", "mean")
    )
    .reset_index()
)

driver_base["realized_price"] = (
    driver_base["total_net_revenue"] /
    driver_base["total_sold"].replace(0, pd.NA)
)

driver_base.head()

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price
0,2 Large Pizza One Topping,783,21234.93,28.23,27.119962
1,2 Liter,2282,2275.48,1.00,0.997143
2,2 Liter Soda,4608,15351.08,3.59,3.331398
3,20 oz Drinks,484,996.16,2.21,2.058182
4,3 Large Pizzas,1,39.70,38.00,39.7


### 1.7b Compute Baseline Revenue at Menu Price

- Objective:
  - Estimate expected revenue if all units sold at menu price.
- Formula:
  - Expected Revenue = total_sold × avg_menu_price

In [167]:
driver_base["expected_menu_revenue"] = (
    driver_base["total_sold"] *
    driver_base["avg_menu_price"]
)

driver_base.head()

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price,expected_menu_revenue
0,2 Large Pizza One Topping,783,21234.93,28.23,27.119962,22104.09
1,2 Liter,2282,2275.48,1.00,0.997143,2282.0
2,2 Liter Soda,4608,15351.08,3.59,3.331398,16542.72
3,20 oz Drinks,484,996.16,2.21,2.058182,1069.64
4,3 Large Pizzas,1,39.70,38.00,39.7,38.0


### 1.7c Decompose Revenue Gap (Price Effect)

- Objective:
  - Measure revenue lost/gained due to pricing deviation.
- Formula:
  - Price Effect = total_net_revenue − expected_menu_revenue

In [168]:
driver_base["price_effect"] = (
    driver_base["total_net_revenue"] -
    driver_base["expected_menu_revenue"]
)

driver_base.head()

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price,expected_menu_revenue,price_effect
0,2 Large Pizza One Topping,783,21234.93,28.23,27.119962,22104.09,-869.16
1,2 Liter,2282,2275.48,1.00,0.997143,2282.0,-6.52
2,2 Liter Soda,4608,15351.08,3.59,3.331398,16542.72,-1191.64
3,20 oz Drinks,484,996.16,2.21,2.058182,1069.64,-73.48
4,3 Large Pizzas,1,39.70,38.00,39.7,38.0,1.7


### 1.7d Compute Contribution Share of Price Effect

- Objective:
  - Rank items by absolute price-driven revenue impact.
- Output:
  - Price effect contribution %

In [169]:
total_price_effect = driver_base["price_effect"].sum()

driver_base["price_effect_share"] = (
    driver_base["price_effect"] /
    total_price_effect if total_price_effect != 0 else 0
)

driver_base = driver_base.sort_values(
    "price_effect",
    ascending=False
)

driver_base.head(20)

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price,expected_menu_revenue,price_effect,price_effect_share
46,Lrg Multi Topping,1590,31150.88,15.22,19.591748,24199.8,6951.08,-1.643472
67,Medium Multi Topping,1169,18156.68,11.49,15.531805,13431.81,4724.87,-1.11712
141,XS Pizza,2986,25638.65,7.17,8.586286,21409.62,4229.03,-0.999887
136,XL Multi Topping,622,14890.19,17.93,23.939212,11152.46,3737.73,-0.883726
108,Small Multi Topping,810,9456.15,8.78,11.674259,7111.8,2344.35,-0.554284
60,Med Half and Half,464,6805.86,10.58,14.667802,4909.12,1896.74,-0.448454
91,SL Multi Topping,138,3992.84,20.92,28.933623,2886.96,1105.88,-0.261468
131,XL Half and Half,542,12017.96,20.32,22.173358,11013.44,1004.52,-0.237503
113,Square Multi Topping,136,3270.07,18.71,24.044632,2544.56,725.51,-0.171535
109,Small Pepperoni Pizza,1097,11469.73,9.93,10.455542,10893.21,576.52,-0.136309


### 1.7e Rank Items by Absolute Price Effect (Dollar Impact)

- Objective:
  - Identify items with the largest pricing-driven revenue distortion.
- Definition:
  - abs_price_effect = |price_effect|
- Output:
  - Top items by absolute dollar impact (regardless of direction)

In [170]:
driver_base = driver_base.copy()

driver_base["abs_price_effect"] = driver_base["price_effect"].abs()

abs_impact_top = (
    driver_base
    .sort_values("abs_price_effect", ascending=False)
    .head(25)
)

abs_impact_top

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price,expected_menu_revenue,price_effect,price_effect_share,abs_price_effect
47,Lrg Pepperoni Pizza,4569,79612.81,20.46,17.424559,93481.74,-13868.93,3.279087,13868.93
46,Lrg Multi Topping,1590,31150.88,15.22,19.591748,24199.8,6951.08,-1.643472,6951.08
67,Medium Multi Topping,1169,18156.68,11.49,15.531805,13431.81,4724.87,-1.11712,4724.87
141,XS Pizza,2986,25638.65,7.17,8.586286,21409.62,4229.03,-0.999887,4229.03
136,XL Multi Topping,622,14890.19,17.93,23.939212,11152.46,3737.73,-0.883726,3737.73
137,XL Pepperoni Pizza,2239,46857.63,22.45,20.927928,50265.55,-3407.92,0.805748,3407.92
119,Wings(10pc),3981,55991.20,14.67,14.064607,58401.27,-2410.07,0.569823,2410.07
23,Family Special (2 Lrg 2 Toppings),7494,228800.60,30.85,30.531172,231189.9,-2389.3,0.564912,2389.3
108,Small Multi Topping,810,9456.15,8.78,11.674259,7111.8,2344.35,-0.554284,2344.35
26,Garlic Bread (12pcs),3822,22596.24,6.48,5.912151,24766.56,-2170.32,0.513137,2170.32


### 1.7f High-Volume Items: Price Compression vs Lift

- Objective:
  - Evaluate pricing distortion among high-volume items.
- Method:
  - Filter to high-volume items using a percentile threshold.
- Output:
  - High-volume subset ranked by price_effect (negative to positive)

In [171]:
volume_threshold = driver_base["total_sold"].quantile(0.90)

volume_threshold

1337.0

In [172]:
high_volume = (
    driver_base
    .loc[driver_base["total_sold"] >= volume_threshold]
    .sort_values("price_effect")
)

high_volume.head(25)

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price,expected_menu_revenue,price_effect,price_effect_share,abs_price_effect
47,Lrg Pepperoni Pizza,4569,79612.81,20.46,17.424559,93481.74,-13868.93,3.279087,13868.93
137,XL Pepperoni Pizza,2239,46857.63,22.45,20.927928,50265.55,-3407.92,0.805748,3407.92
119,Wings(10pc),3981,55991.20,14.67,14.064607,58401.27,-2410.07,0.569823,2410.07
23,Family Special (2 Lrg 2 Toppings),7494,228800.60,30.85,30.531172,231189.9,-2389.3,0.564912,2389.3
26,Garlic Bread (12pcs),3822,22596.24,6.48,5.912151,24766.56,-2170.32,0.513137,2170.32
36,Lrg Combination Pizza,3037,73447.64,24.60,24.184274,74710.2,-1262.56,0.298512,1262.56
2,2 Liter Soda,4608,15351.08,3.59,3.331398,16542.72,-1191.64,0.281744,1191.64
16,Cheese Bread,1794,10640.79,6.48,5.931321,11625.12,-984.33,0.232729,984.33
14,Canned Soda,2204,2838.84,1.47,1.28804,3239.88,-401.04,0.094819,401.04
55,Med Combination Pizza,1425,27524.04,19.49,19.315116,27773.25,-249.21,0.058922,249.21


### 1.7g High-Volume Items: Summary Diagnostics

- Objective:
  - Provide interpretable columns for high-volume pricing analysis.
- Output:
  - total_sold
  - avg_menu_price
  - realized_price
  - price_effect
  - abs_price_effect
  - price_effect_pct_of_menu (context)

In [173]:
high_volume_diag = high_volume.copy()

high_volume_diag["price_effect_per_unit"] = (
    high_volume_diag["price_effect"] /
    high_volume_diag["total_sold"].replace(0, pd.NA)
)

high_volume_diag = high_volume_diag[
    [
        "Name",
        "total_sold",
        "avg_menu_price",
        "realized_price",
        "price_effect",
        "abs_price_effect",
        "price_effect_per_unit"
    ]
]

high_volume_diag.head(25)

,Name,total_sold,avg_menu_price,realized_price,price_effect,abs_price_effect,price_effect_per_unit
47,Lrg Pepperoni Pizza,4569,20.46,17.424559,-13868.93,13868.93,-3.035441
137,XL Pepperoni Pizza,2239,22.45,20.927928,-3407.92,3407.92,-1.522072
119,Wings(10pc),3981,14.67,14.064607,-2410.07,2410.07,-0.605393
23,Family Special (2 Lrg 2 Toppings),7494,30.85,30.531172,-2389.3,2389.3,-0.318828
26,Garlic Bread (12pcs),3822,6.48,5.912151,-2170.32,2170.32,-0.567849
36,Lrg Combination Pizza,3037,24.60,24.184274,-1262.56,1262.56,-0.415726
2,2 Liter Soda,4608,3.59,3.331398,-1191.64,1191.64,-0.258602
16,Cheese Bread,1794,6.48,5.931321,-984.33,984.33,-0.548679
14,Canned Soda,2204,1.47,1.28804,-401.04,401.04,-0.18196
55,Med Combination Pizza,1425,19.49,19.315116,-249.21,249.21,-0.174884


## 1.8 Revenue Concentration × Pricing Interaction

- Objective:
  - Identify whether pricing compression affects core revenue drivers.
- Definitions:
  - Revenue Share = total_net_revenue / total portfolio net revenue
  - Cumulative Share = running total of revenue share
- Output:
  - Pareto ranking (80/20)
  - Overlay with price_effect

### 1.8a Compute Revenue Share

In [174]:
portfolio_net_revenue = driver_base["total_net_revenue"].sum()

driver_base = driver_base.copy()

driver_base["revenue_share"] = (
    driver_base["total_net_revenue"] /
    portfolio_net_revenue
)

driver_base = driver_base.sort_values(
    "total_net_revenue",
    ascending=False
)

driver_base["cumulative_revenue_share"] = (
    driver_base["revenue_share"].cumsum()
)

driver_base.head(20)

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price,expected_menu_revenue,price_effect,price_effect_share,abs_price_effect,revenue_share,cumulative_revenue_share
23,Family Special (2 Lrg 2 Toppings),7494,228800.60,30.85,30.531172,231189.9,-2389.3,0.564912,2389.3,0.184392,0.184392
47,Lrg Pepperoni Pizza,4569,79612.81,20.46,17.424559,93481.74,-13868.93,3.279087,13868.93,0.064161,0.248553
36,Lrg Combination Pizza,3037,73447.64,24.60,24.184274,74710.2,-1262.56,0.298512,1262.56,0.059192,0.307745
119,Wings(10pc),3981,55991.20,14.67,14.064607,58401.27,-2410.07,0.569823,2410.07,0.045124,0.352869
137,XL Pepperoni Pizza,2239,46857.63,22.45,20.927928,50265.55,-3407.92,0.805748,3407.92,0.037763,0.390632
126,XL Combination Pizza,1337,36549.69,27.08,27.337091,36205.96,343.73,-0.081269,343.73,0.029456,0.420088
68,Medium Pepperoni Pizza,2250,31354.82,NaN,13.935476,<NA>,<NA>,<NA>,<NA>,0.025269,0.445357
46,Lrg Multi Topping,1590,31150.88,15.22,19.591748,24199.8,6951.08,-1.643472,6951.08,0.025105,0.470462
55,Med Combination Pizza,1425,27524.04,19.49,19.315116,27773.25,-249.21,0.058922,249.21,0.022182,0.492644
22,Delivery,6619,26456.00,4.00,3.996978,26476.0,-20.0,0.004729,20.0,0.021321,0.513965


### 1.8b Identify Core Revenue Set (Pareto 80%)

- Objective:
  - Isolate items contributing to first 80% of net revenue.
- Rule:
  - cumulative_revenue_share ≤ 0.80

In [175]:
core_revenue_set = (
    driver_base
    .loc[driver_base["cumulative_revenue_share"] <= 0.80]
)

core_revenue_set.head(20)

,Name,total_sold,total_net_revenue,avg_menu_price,realized_price,expected_menu_revenue,price_effect,price_effect_share,abs_price_effect,revenue_share,cumulative_revenue_share
23,Family Special (2 Lrg 2 Toppings),7494,228800.60,30.85,30.531172,231189.9,-2389.3,0.564912,2389.3,0.184392,0.184392
47,Lrg Pepperoni Pizza,4569,79612.81,20.46,17.424559,93481.74,-13868.93,3.279087,13868.93,0.064161,0.248553
36,Lrg Combination Pizza,3037,73447.64,24.60,24.184274,74710.2,-1262.56,0.298512,1262.56,0.059192,0.307745
119,Wings(10pc),3981,55991.20,14.67,14.064607,58401.27,-2410.07,0.569823,2410.07,0.045124,0.352869
137,XL Pepperoni Pizza,2239,46857.63,22.45,20.927928,50265.55,-3407.92,0.805748,3407.92,0.037763,0.390632
126,XL Combination Pizza,1337,36549.69,27.08,27.337091,36205.96,343.73,-0.081269,343.73,0.029456,0.420088
68,Medium Pepperoni Pizza,2250,31354.82,NaN,13.935476,<NA>,<NA>,<NA>,<NA>,0.025269,0.445357
46,Lrg Multi Topping,1590,31150.88,15.22,19.591748,24199.8,6951.08,-1.643472,6951.08,0.025105,0.470462
55,Med Combination Pizza,1425,27524.04,19.49,19.315116,27773.25,-249.21,0.058922,249.21,0.022182,0.492644
22,Delivery,6619,26456.00,4.00,3.996978,26476.0,-20.0,0.004729,20.0,0.021321,0.513965


### 1.8c Overlay Pricing Effect on Core Revenue Items

- Objective:
  - Evaluate price_effect within core revenue set.
- Output:
  - total_sold
  - realized_price
  - avg_menu_price
  - price_effect
  - revenue_share

In [176]:
core_pricing_overlay = core_revenue_set[
    [
        "Name",
        "total_net_revenue",
        "revenue_share",
        "total_sold",
        "avg_menu_price",
        "realized_price",
        "price_effect",
        "abs_price_effect"
    ]
].sort_values("price_effect")

core_pricing_overlay.head(25)

,Name,total_net_revenue,revenue_share,total_sold,avg_menu_price,realized_price,price_effect,abs_price_effect
47,Lrg Pepperoni Pizza,79612.81,0.064161,4569,20.46,17.424559,-13868.93,13868.93
137,XL Pepperoni Pizza,46857.63,0.037763,2239,22.45,20.927928,-3407.92,3407.92
119,Wings(10pc),55991.20,0.045124,3981,14.67,14.064607,-2410.07,2410.07
23,Family Special (2 Lrg 2 Toppings),228800.60,0.184392,7494,30.85,30.531172,-2389.3,2389.3
26,Garlic Bread (12pcs),22596.24,0.018211,3822,6.48,5.912151,-2170.32,2170.32
36,Lrg Combination Pizza,73447.64,0.059192,3037,24.60,24.184274,-1262.56,1262.56
2,2 Liter Soda,15351.08,0.012372,4608,3.59,3.331398,-1191.64,1191.64
16,Cheese Bread,10640.79,0.008576,1794,6.48,5.931321,-984.33,984.33
0,2 Large Pizza One Topping,21234.93,0.017113,783,28.23,27.119962,-869.16,869.16
41,Lrg Half and Half,17882.80,0.014412,988,18.85,18.1,-741.0,741.0


### 1.8d Flag Core Revenue Items with Negative Price Effect

- Objective:
  - Identify margin compression within top revenue drivers.
- Criteria:
  - price_effect < 0

In [177]:
core_negative_pricing = (
    core_pricing_overlay
    .loc[core_pricing_overlay["price_effect"] < 0]
    .sort_values("price_effect")
)

core_negative_pricing.head(25)

,Name,total_net_revenue,revenue_share,total_sold,avg_menu_price,realized_price,price_effect,abs_price_effect
47,Lrg Pepperoni Pizza,79612.81,0.064161,4569,20.46,17.424559,-13868.93,13868.93
137,XL Pepperoni Pizza,46857.63,0.037763,2239,22.45,20.927928,-3407.92,3407.92
119,Wings(10pc),55991.20,0.045124,3981,14.67,14.064607,-2410.07,2410.07
23,Family Special (2 Lrg 2 Toppings),228800.60,0.184392,7494,30.85,30.531172,-2389.3,2389.3
26,Garlic Bread (12pcs),22596.24,0.018211,3822,6.48,5.912151,-2170.32,2170.32
36,Lrg Combination Pizza,73447.64,0.059192,3037,24.60,24.184274,-1262.56,1262.56
2,2 Liter Soda,15351.08,0.012372,4608,3.59,3.331398,-1191.64,1191.64
16,Cheese Bread,10640.79,0.008576,1794,6.48,5.931321,-984.33,984.33
0,2 Large Pizza One Topping,21234.93,0.017113,783,28.23,27.119962,-869.16,869.16
41,Lrg Half and Half,17882.80,0.014412,988,18.85,18.1,-741.0,741.0


### 1.8e Quantify Core Revenue at Risk

- Objective:
  - Measure total net revenue and share exposed to negative price_effect within core revenue set.
- Outputs:
  - Core revenue total
  - Revenue under compression
  - % of core revenue compressed
  - % of total portfolio compressed

In [178]:
core_total_revenue = core_revenue_set["total_net_revenue"].sum()

core_negative_revenue = core_negative_pricing["total_net_revenue"].sum()

portfolio_total_revenue = driver_base["total_net_revenue"].sum()

core_negative_pct_of_core = core_negative_revenue / core_total_revenue
core_negative_pct_of_portfolio = core_negative_revenue / portfolio_total_revenue

core_total_revenue, core_negative_revenue, core_negative_pct_of_core, core_negative_pct_of_portfolio

(983471.72, 692027.8, 0.7036580573969123, 0.5577114607406004)

## 1.9 Category-Level Revenue Driver Decomposition

- Objective:
  - Evaluate pricing distortion at category level.
- Definitions:
  - Net Revenue = Total Revenue − Discounts
  - Expected Menu Revenue = total_sold × avg_menu_price
  - Price Effect = Net Revenue − Expected Menu Revenue
- Output:
  - Category-level price compression/lift
  - Absolute dollar impact
  - Revenue share context

### 1.9a Aggregate to Category Level

In [179]:
category_driver = (
    item_2025_merged
    .assign(net_revenue=lambda df: df["Total Revenue"] - df["Discounts"])
    .groupby("Category Name")
    .agg(
        total_sold=("Sold", "sum"),
        total_net_revenue=("net_revenue", "sum"),
        avg_menu_price=("Menu Price", "mean")
    )
    .reset_index()
)

category_driver["expected_menu_revenue"] = (
    category_driver["total_sold"] *
    category_driver["avg_menu_price"]
)

category_driver["price_effect"] = (
    category_driver["total_net_revenue"] -
    category_driver["expected_menu_revenue"]
)

category_driver["abs_price_effect"] = (
    category_driver["price_effect"].abs()
)

category_driver.head()

,Category Name,total_sold,total_net_revenue,avg_menu_price,expected_menu_revenue,price_effect,abs_price_effect
0,Additional Sauces,23192,11138.75,0.675,15654.6,-4515.85,4515.85
1,All Meat Pizza,1022,23604.10,23.898,24423.756,-819.656,819.656
2,BBQ Chicken Pizza,487,11040.40,23.898,11638.326,-597.926,597.926
3,Bacon Chicken Pizza,647,14787.27,23.898,15462.006,-674.736,674.736
4,Cheese Pizza,2209,27791.45,15.515,34272.635,-6481.185,6481.185


### 1.9b Rank Categories by Absolute Price Effect

- Objective:
  - Identify which categories drive the largest pricing distortion in dollars.
- Output:
  - Ranked by abs_price_effect

In [180]:
category_driver_ranked = (
    category_driver
    .sort_values("abs_price_effect", ascending=False)
)

category_driver_ranked

,Category Name,total_sold,total_net_revenue,avg_menu_price,expected_menu_revenue,price_effect,abs_price_effect
21,Pepperoni Pizza,12069,193467.53,17.640000,212897.16,-19429.63,19429.63
11,Full Build Your Own Pizza,4465,80916.81,15.508333,69244.708333,11672.101667,11672.101667
25,Uncategorized,2033,20633.37,4.410000,8965.53,11667.84,11667.84
24,Sides,14277,125735.16,8.094286,115562.117143,10173.042857,10173.042857
6,Combination Pizza,6858,156166.11,23.898000,163892.484,-7726.374,7726.374
4,Cheese Pizza,2209,27791.45,15.515000,34272.635,-6481.185,6481.185
0,Additional Sauces,23192,11138.75,0.675000,15654.6,-4515.85,4515.85
27,XS Pizza,2986,25638.65,7.170000,21409.62,4229.03,4229.03
17,Half and Half Build Your Own Pizza,2170,41205.78,17.152500,37220.925,3984.855,3984.855
23,Salads,2416,15449.37,7.138000,17245.408,-1796.038,1796.038


### 1.9c Compute Category Revenue Share and Compression %

- Objective:
  - Contextualize price effect relative to category revenue.
- Formulas:
  - Revenue Share = category_net_revenue / portfolio_net_revenue
  - Price Effect % = price_effect / expected_menu_revenue

In [181]:
portfolio_net_revenue = driver_base["total_net_revenue"].sum()

category_driver_ranked["revenue_share"] = (
    category_driver_ranked["total_net_revenue"] /
    portfolio_net_revenue
)

category_driver_ranked["price_effect_pct"] = (
    category_driver_ranked["price_effect"] /
    category_driver_ranked["expected_menu_revenue"].replace(0, pd.NA)
)

category_driver_ranked

,Category Name,total_sold,total_net_revenue,avg_menu_price,expected_menu_revenue,price_effect,abs_price_effect,revenue_share,price_effect_pct
21,Pepperoni Pizza,12069,193467.53,17.640000,212897.16,-19429.63,19429.63,0.155917,-0.091263
11,Full Build Your Own Pizza,4465,80916.81,15.508333,69244.708333,11672.101667,11672.101667,0.065212,0.168563
25,Uncategorized,2033,20633.37,4.410000,8965.53,11667.84,11667.84,0.016629,1.301411
24,Sides,14277,125735.16,8.094286,115562.117143,10173.042857,10173.042857,0.101331,0.088031
6,Combination Pizza,6858,156166.11,23.898000,163892.484,-7726.374,7726.374,0.125856,-0.047143
4,Cheese Pizza,2209,27791.45,15.515000,34272.635,-6481.185,6481.185,0.022397,-0.189107
0,Additional Sauces,23192,11138.75,0.675000,15654.6,-4515.85,4515.85,0.008977,-0.288468
27,XS Pizza,2986,25638.65,7.170000,21409.62,4229.03,4229.03,0.020662,0.197529
17,Half and Half Build Your Own Pizza,2170,41205.78,17.152500,37220.925,3984.855,3984.855,0.033208,0.10706
23,Salads,2416,15449.37,7.138000,17245.408,-1796.038,1796.038,0.012451,-0.104146


### 1.9d Executive Summary — Portfolio Compression Metrics

- Objective:
  - Quantify total category-level pricing compression.
- Definitions:
  - Negative Price Effect Categories = price_effect < 0
  - Total Compression = sum of negative category price_effect
- Outputs:
  - Total compressed dollars
  - % of total portfolio revenue affected

In [182]:
# Identify compressed categories
compressed_categories = category_driver_ranked.loc[
    category_driver_ranked["price_effect"] < 0
]

# Total compression (absolute dollars)
total_category_compression = (
    compressed_categories["price_effect"].sum()
)

# Portfolio total revenue (already computed earlier)
portfolio_total_revenue = driver_base["total_net_revenue"].sum()

# Compression as % of portfolio
compression_pct_of_portfolio = (
    total_category_compression /
    portfolio_total_revenue
)

total_category_compression, compression_pct_of_portfolio

(-48508.57607142861, -0.03909350002881818)

## 1.10 Executive Summary — Pricing Architecture Findings

- Scope:
  - SKU-level, core revenue, and category-level pricing decomposition completed.

- Portfolio-Level Compression:
  - ~$48.5K negative price effect at category level.
  - ~3.9% of total portfolio net revenue.

- Core Revenue Risk:
  - Largest revenue-driving categories (e.g., Pepperoni Pizza) exhibit structural price compression.
  - Compression is concentrated in traditional pizza SKUs.

- Structural Offsets:
  - Positive price effects observed in:
    - Full Build Your Own Pizza
    - Sides
    - Modifier-driven items
  - Upsell architecture partially offsets core pizza compression.

- Strategic Insight:
  - Business is not broadly underpriced.
  - Compression is SKU- and category-specific.
  - Primary focus area:
    - Core pizza pricing discipline (discounting, promotions, overrides).

- Conclusion:
  - Pricing system is functional but not optimized.
  - Targeted adjustment in core pizza categories could improve margin without structural redesign.

## 2. Strategic Revenue Levers — Post-Diagnostic Layer
- Transition from pricing diagnostics → controllable revenue action.
- Constrain scope using revenue concentration (Pareto discipline).
- Every output must directly support prioritization.
- No repetition of prior diagnostics.

### 2.0 Section 2 Inputs — Define `order_items`
- Section 2 expects a transaction-level table named `order_items`.
- Source table in this workbook: `item_2025_merged`.
- Definitions:
  - realized_revenue = Total Revenue - Discounts
  - SKU = SKU if present else Name
  - Category = Category Name

In [183]:
# --- Require upstream table ---
try:
    item_2025_merged
except NameError:
    raise NameError("item_2025_merged is not defined. Run Section 1 merge step before Section 2.")

# --- Require columns needed to define order_items ---
required_cols = ["Name", "Total Revenue", "Discounts", "Category Name"]
missing_cols = [c for c in required_cols if c not in item_2025_merged.columns]

if missing_cols:
    raise KeyError(f"item_2025_merged missing required columns: {missing_cols}")

# --- Build order_items (thin, deterministic view) ---
order_items = item_2025_merged.copy()

order_items["Total Revenue"] = pd.to_numeric(order_items["Total Revenue"], errors="coerce").fillna(0)
order_items["Discounts"] = pd.to_numeric(order_items["Discounts"], errors="coerce").fillna(0)

order_items["realized_revenue"] = order_items["Total Revenue"] - order_items["Discounts"]

order_items["Category"] = order_items["Category Name"].fillna("UNKNOWN_CATEGORY")

if "SKU" not in order_items.columns:
    order_items["SKU"] = pd.NA

order_items["SKU"] = order_items["SKU"].fillna(order_items["Name"]).fillna("UNKNOWN_SKU")

order_items[["Name", "SKU", "Category", "Total Revenue", "Discounts", "realized_revenue"]].head(10)

,Name,SKU,Category,Total Revenue,Discounts,realized_revenue
0,Family Special (2 Lrg 2 Toppings),Family Special (2 Lrg 2 Toppings),Deals,228671.21,-129.39,228800.60
1,Three Seasons (XL),Three Seasons (XL),Deals,23743.58,-4.87,23748.45
2,Large Pizza & Bread Sticks,Large Pizza & Bread Sticks,Deals,21607.20,-14.42,21621.62
3,2 Large Pizza One Topping,2 Large Pizza One Topping,Deals,21214.72,-20.21,21234.93
4,Pizza Party Pack (Medium Pizza and 6 Wings),Pizza Party Pack (Medium Pizza and 6 Wings),Deals,8004.79,-6.98,8011.77
5,"Superbowl deal (XL, 10 wings, GB)","Superbowl deal (XL, 10 wings, GB)",Deals,110.34,0.00,110.34
6,3 Large Pizzas,3 Large Pizzas,Deals,39.70,0.00,39.70
7,Lrg Pepperoni Pizza,Lrg Pepperoni Pizza,Pepperoni Pizza,79413.93,-198.88,79612.81
8,XL Pepperoni Pizza,XL Pepperoni Pizza,Pepperoni Pizza,46673.48,-184.15,46857.63
9,Medium Pepperoni Pizza,Medium Pepperoni Pizza,Pepperoni Pizza,31254.92,-99.90,31354.82


### 2.1 Revenue Concentration Mapping
- Identify which SKUs and Categories control the majority of realized revenue.
- Compute revenue share and cumulative concentration.
- Flag entities contributing to the first 80% of portfolio revenue.
- Produce reusable focus sets for downstream strategy sections.

### 2.1.1 Validate Required Inputs
- Require: realized_revenue, SKU, Category in `order_items`.
- Fill missing SKU/Category with sentinel values to prevent groupby loss.
- Fail fast if required columns are missing.

In [184]:
required_cols = {"realized_revenue", "SKU", "Category"}
missing = required_cols - set(order_items.columns)

if missing:
    raise KeyError(f"order_items missing required columns: {sorted(missing)}")

order_items["SKU"] = order_items["SKU"].fillna("UNKNOWN_SKU")
order_items["Category"] = order_items["Category"].fillna("UNKNOWN_CATEGORY")

#### 2.1.2 Discount Sign Sanity Check
- Discounts are expected to be mostly negative (Clover convention).
- If discounts are mostly positive, realized_revenue formula must be revisited.

In [185]:
discounts = order_items["Discounts"]

discount_sign_summary = pd.DataFrame({
    "sign_bucket": ["negative", "zero", "positive"],
    "count": [
        int((discounts < 0).sum()),
        int((discounts == 0).sum()),
        int((discounts > 0).sum())
    ]
})

discount_sign_summary["share"] = (
    discount_sign_summary["count"] /
    discount_sign_summary["count"].sum()
)

discount_sign_summary

,sign_bucket,count,share
0,negative,93,0.164021
1,zero,474,0.835979
2,positive,0,0.000000


### 2.1.3 Portfolio Total (Realized Revenue)
- Compute total realized revenue once for share calculations.
- Fail fast if total is non-positive.

In [186]:
portfolio_total_revenue = float(order_items["realized_revenue"].sum())

if portfolio_total_revenue <= 0:
    raise ValueError("Portfolio total realized_revenue must be > 0 for concentration analysis.")

### 2.1.4 SKU-Level Revenue Concentration Table
- Aggregate realized revenue by SKU.
- Rank descending by realized revenue.
- Compute revenue_share and cumulative_share.
- Flag SKUs contributing to first 80% of portfolio revenue.
- Preview top 20 SKUs by realized revenue.

In [187]:
sku_revenue = (
    order_items
    .groupby("SKU", as_index=False)
    .agg(realized_revenue=("realized_revenue", "sum"))
    .sort_values("realized_revenue", ascending=False)
    .reset_index(drop=True)
)

sku_revenue["revenue_share"] = sku_revenue["realized_revenue"] / portfolio_total_revenue
sku_revenue["cumulative_share"] = sku_revenue["revenue_share"].cumsum()
sku_revenue["top_80_flag"] = sku_revenue["cumulative_share"] <= 0.80

sku_revenue.head(20)

,SKU,realized_revenue,revenue_share,cumulative_share,top_80_flag
0,Family Special (2 Lrg 2 Toppings),228800.60,0.184392,0.184392,True
1,Lrg Pepperoni Pizza,79612.81,0.064161,0.248553,True
2,Lrg Combination Pizza,73447.64,0.059192,0.307745,True
3,Wings(10pc),55991.20,0.045124,0.352869,True
4,XL Pepperoni Pizza,46857.63,0.037763,0.390632,True
5,XL Combination Pizza,36549.69,0.029456,0.420088,True
6,Medium Pepperoni Pizza,31354.82,0.025269,0.445357,True
7,Lrg Multi Topping,31150.88,0.025105,0.470462,True
8,Med Combination Pizza,27524.04,0.022182,0.492644,True
9,Delivery,26456.00,0.021321,0.513965,True


### 2.1.5 Category-Level Revenue Concentration Table
- Aggregate realized revenue by Category.
- Rank descending by realized revenue.
- Compute revenue_share and cumulative_share.
- Flag Categories contributing to first 80% of portfolio revenue.

In [188]:
category_revenue = (
    order_items
    .groupby("Category", as_index=False)
    .agg(realized_revenue=("realized_revenue", "sum"))
    .sort_values("realized_revenue", ascending=False)
    .reset_index(drop=True)
)

category_revenue["revenue_share"] = category_revenue["realized_revenue"] / portfolio_total_revenue
category_revenue["cumulative_share"] = category_revenue["revenue_share"].cumsum()
category_revenue["top_80_flag"] = category_revenue["cumulative_share"] <= 0.80

category_revenue

,Category,realized_revenue,revenue_share,cumulative_share,top_80_flag
0,Deals,303567.41,0.244648,0.244648,True
1,Pepperoni Pizza,193467.53,0.155917,0.400565,True
2,Combination Pizza,156166.11,0.125856,0.526421,True
3,Sides,125735.16,0.101331,0.627752,True
4,Full Build Your Own Pizza,80916.81,0.065212,0.692963,True
5,Hawaiian Delight Pizza,45053.56,0.036309,0.729272,True
6,Half and Half Build Your Own Pizza,41205.78,0.033208,0.762481,True
7,Garlic Chicken Pizza,33745.20,0.027196,0.789676,True
8,Cheese Pizza,27791.45,0.022397,0.812073,False
9,Delivery,26456.00,0.021321,0.833395,False


### 2.1.6 Concentration Summary Metrics
- Compute how many SKUs generate the first 80% of revenue.
- Compute how many Categories generate the first 80% of revenue.
- Compute entity share (concentration density) for each level.

In [189]:
sku_80_count = int(sku_revenue["top_80_flag"].sum())
sku_total_count = int(len(sku_revenue))

category_80_count = int(category_revenue["top_80_flag"].sum())
category_total_count = int(len(category_revenue))

concentration_summary = pd.DataFrame({
    "level": ["SKU", "Category"],
    "top_80_count": [sku_80_count, category_80_count],
    "total_count": [sku_total_count, category_total_count]
})

concentration_summary["share_of_entities"] = (
    concentration_summary["top_80_count"] / concentration_summary["total_count"]
)

concentration_summary

,level,top_80_count,total_count,share_of_entities
0,SKU,32,181,0.176796
1,Category,8,28,0.285714


### 2.1.7 Export Focus Sets for Section 2.2+
- Create reusable focus sets for downstream strategy analysis.
- Default downstream sections to these sets unless explicitly widened.
- Output sizes to confirm scope width.

In [190]:
focus_skus_top80 = set(sku_revenue.loc[sku_revenue["top_80_flag"], "SKU"])
focus_categories_top80 = set(category_revenue.loc[category_revenue["top_80_flag"], "Category"])

len(focus_skus_top80), len(focus_categories_top80)

(32, 8)

## 2.2 Discount Structure Impact Analysis
- Quantify how discounts affect realized revenue across the portfolio.
- Identify which SKUs absorb the largest discount pressure.
- Determine whether discounts are concentrated or broadly distributed.
- Restrict analysis to revenue-driving entities where appropriate.

### 2.2.1 Validate Required Inputs
- Require: realized_revenue, SKU, Category in `order_items`.
- Fill missing SKU/Category with sentinel values to prevent groupby loss.
- Fail fast if required columns are missing.

In [191]:
required_cols = {"realized_revenue", "SKU", "Category"}
missing = required_cols - set(order_items.columns)

if missing:
    raise KeyError(f"order_items missing required columns: {sorted(missing)}")

order_items["SKU"] = order_items["SKU"].fillna("UNKNOWN_SKU")
order_items["Category"] = order_items["Category"].fillna("UNKNOWN_CATEGORY")

### 2.2.2 Discount Sign Sanity Check
- Discounts are expected to be mostly negative (Clover convention).
- If discounts are mostly positive, realized_revenue formula must be revisited.

In [192]:
discounts = order_items["Discounts"]

discount_sign_summary = pd.DataFrame({
    "sign_bucket": ["negative", "zero", "positive"],
    "count": [
        int((discounts < 0).sum()),
        int((discounts == 0).sum()),
        int((discounts > 0).sum())
    ]
})

discount_sign_summary["share"] = discount_sign_summary["count"] / discount_sign_summary["count"].sum()

discount_sign_summary

,sign_bucket,count,share
0,negative,93,0.164021
1,zero,474,0.835979
2,positive,0,0.000000


### 2.2.3 Portfolio Discount Impact
- Compute total discounts across the dataset.
- Compare discounts to realized revenue.
- Estimate portfolio-level discount rate.

In [193]:
total_discounts = order_items["Discounts"].sum()
total_realized_revenue = order_items["realized_revenue"].sum()

portfolio_discount_rate = abs(total_discounts) / total_realized_revenue

discount_overview = pd.DataFrame({
    "metric": ["total_discounts", "total_realized_revenue", "portfolio_discount_rate"],
    "value": [total_discounts, total_realized_revenue, portfolio_discount_rate]
})

discount_overview

,metric,value
0,total_discounts,-5.434300e+03
1,total_realized_revenue,1.240835e+06
2,portfolio_discount_rate,4.379552e-03


#### 2.2.4 Discount Impact by SKU
- Aggregate discounts at the SKU level.
- Measure discount pressure relative to realized revenue.
- Identify SKUs absorbing the largest discount exposure.

In [194]:
sku_discount = (
    order_items
    .groupby("SKU", as_index=False)
    .agg(
        total_discounts=("Discounts", "sum"),
        realized_revenue=("realized_revenue", "sum")
    )
)

sku_discount["discount_rate"] = (
    sku_discount["total_discounts"].abs() /
    sku_discount["realized_revenue"]
)

sku_discount = sku_discount.sort_values(
    "total_discounts",
    ascending=True
).reset_index(drop=True)

sku_discount.head(20)

,SKU,total_discounts,realized_revenue,discount_rate
0,XL Garlic Chicken Pizza,-631.55,12823.79,0.049248
1,XL Combination Pizza,-563.53,36549.69,0.015418
2,Lrg Garlic Chicken Pizza,-317.31,12947.23,0.024508
3,Lrg Combination Pizza,-231.31,73447.64,0.003149
4,Lrg Pepperoni Pizza,-198.88,79612.81,0.002498
5,XL Pepperoni Pizza,-184.15,46857.63,0.003930
6,Wings(10pc),-180.59,55991.20,0.003225
7,Medium Multi Topping,-150.20,18156.68,0.008272
8,XL Chef's Special Pizza,-134.50,2135.65,0.062978
9,Family Special (2 Lrg 2 Toppings),-129.39,228800.60,0.000566


### 2.2.5 Discount Impact by Category
- Aggregate discounts at the category level.
- Compute category-level discount rates.
- Identify menu segments most affected by discounting.

In [195]:
category_discount = (
    order_items
    .groupby("Category", as_index=False)
    .agg(
        total_discounts=("Discounts", "sum"),
        realized_revenue=("realized_revenue", "sum")
    )
)

category_discount["discount_rate"] = (
    category_discount["total_discounts"].abs() /
    category_discount["realized_revenue"]
)

category_discount = category_discount.sort_values(
    "total_discounts",
    ascending=True
).reset_index(drop=True)

category_discount

,Category,total_discounts,realized_revenue,discount_rate
0,Garlic Chicken Pizza,-1053.10,33745.20,0.031207
1,Combination Pizza,-932.56,156166.11,0.005972
2,Pepperoni Pizza,-717.44,193467.53,0.003708
3,Sides,-659.86,125735.16,0.005248
4,Full Build Your Own Pizza,-447.12,80916.81,0.005526
5,Hawaiian Delight Pizza,-258.17,45053.56,0.005730
6,Bacon Chicken Pizza,-215.14,14787.27,0.014549
7,Deals,-175.87,303567.41,0.000579
8,BBQ Chicken Pizza,-167.46,11040.40,0.015168
9,Chef's Special Pizza,-151.23,7231.46,0.020913


### 2.2.6 Discount Concentration Summary
- Summarize how widely discounts are distributed across SKUs and categories.
- Measure how many entities receive discounts.
- Prepare inputs for strategic prioritization in later sections.

In [196]:
sku_with_discounts = int((sku_discount["total_discounts"] != 0).sum())
category_with_discounts = int((category_discount["total_discounts"] != 0).sum())

discount_summary = pd.DataFrame({
    "metric": [
        "portfolio_discount_rate",
        "sku_with_discounts",
        "categories_with_discounts"
    ],
    "value": [
        portfolio_discount_rate,
        sku_with_discounts,
        category_with_discounts
    ]
})

discount_summary

,metric,value
0,portfolio_discount_rate,0.00438
1,sku_with_discounts,91.00000
2,categories_with_discounts,26.00000


### 2.3 SKU Architecture Stress Test
- Identify SKUs where realized pricing diverges materially from expected menu structure.
- Detect SKUs that require persistent discounting to drive volume.
- Focus analysis on revenue-driving SKUs identified in Section 2.1.

### 2.3.1 Restrict Analysis to Revenue-Driving SKUs
- Limit analysis to SKUs contributing to the first 80% of revenue.
- Prevent long-tail SKUs from diluting structural diagnostics.
- Create working dataset for architecture analysis.

In [197]:
architecture_df = order_items[
    order_items["SKU"].isin(focus_skus_top80)
].copy()

architecture_df.shape

(354, 42)

### 2.3.2 Aggregate SKU Pricing Structure
- Aggregate realized revenue and discounts by SKU.
- Count transactions per SKU.
- Estimate average realized revenue per transaction.
- Prepare base table for architecture diagnostics.

In [200]:
sku_architecture = (
    architecture_df
    .groupby("SKU", as_index=False)
    .agg(
        realized_revenue=("realized_revenue", "sum"),
        total_discounts=("Discounts", "sum"),
        order_lines=("SKU", "count")
    )
)

sku_architecture["avg_realized_revenue_per_txn"] = (
    sku_architecture["realized_revenue"] /
    sku_architecture["order_lines"]
)

sku_architecture.head(20)

,SKU,realized_revenue,total_discounts,order_lines,avg_realized_revenue_per_txn
0,2 Large Pizza One Topping,21234.93,-20.21,1,21234.930000
1,2 Liter Soda,15351.08,-54.06,1,15351.080000
2,Cheese Bread,10640.79,-68.88,1,10640.790000
3,Delivery,26456.00,-18.31,1,26456.000000
4,Family Special (2 Lrg 2 Toppings),228800.60,-129.39,1,228800.600000
5,Garlic Bread (12pcs),22596.24,-86.26,1,22596.240000
6,Large Pizza & Bread Sticks,21621.62,-14.42,1,21621.620000
7,Lrg All Meat Pizza,9758.34,-8.37,1,9758.340000
8,Lrg Combination Pizza,73447.64,-231.31,1,73447.640000
9,Lrg Garlic Chicken Pizza,12947.23,-317.31,1,12947.230000


### 2.3.3 Discount Pressure by SKU
- Measure discount pressure as discounts relative to realized revenue.
- Identify SKUs that depend heavily on discounting.
- Rank SKUs by discount pressure.

In [201]:
sku_architecture["discount_pressure"] = (
    sku_architecture["total_discounts"].abs() /
    sku_architecture["realized_revenue"]
)

sku_architecture = sku_architecture.sort_values(
    "discount_pressure",
    ascending=False
).reset_index(drop=True)

sku_architecture.head(20)

,SKU,realized_revenue,total_discounts,order_lines,avg_realized_revenue_per_txn,discount_pressure
0,XL Garlic Chicken Pizza,12823.79,-631.55,1,12823.79,0.049248
1,Lrg Garlic Chicken Pizza,12947.23,-317.31,1,12947.23,0.024508
2,XL Combination Pizza,36549.69,-563.53,1,36549.69,0.015418
3,XL Hawaiian Delight Pizza,11683.74,-119.94,1,11683.74,0.010266
4,Small Pepperoni Pizza,11469.73,-104.90,1,11469.73,0.009146
5,XS Pepperoni Pizza,11251.28,-96.22,1,11251.28,0.008552
6,Medium Multi Topping,18156.68,-150.20,1,18156.68,0.008272
7,XL Multi Topping,14890.19,-100.14,1,14890.19,0.006725
8,Cheese Bread,10640.79,-68.88,1,10640.79,0.006473
9,Wings (6pc),9800.48,-55.96,1,9800.48,0.005710


### 2.3.4 Flag High-Pressure SKUs
- Identify SKUs with unusually high discount pressure.
- These represent candidates for menu repricing or promotion restructuring.
- Flag threshold set at 10% discount pressure.

In [202]:
sku_architecture["high_discount_flag"] = (
    sku_architecture["discount_pressure"] > 0.10
)

sku_architecture.head(20)

,SKU,realized_revenue,total_discounts,order_lines,avg_realized_revenue_per_txn,discount_pressure,high_discount_flag
0,XL Garlic Chicken Pizza,12823.79,-631.55,1,12823.79,0.049248,False
1,Lrg Garlic Chicken Pizza,12947.23,-317.31,1,12947.23,0.024508,False
2,XL Combination Pizza,36549.69,-563.53,1,36549.69,0.015418,False
3,XL Hawaiian Delight Pizza,11683.74,-119.94,1,11683.74,0.010266,False
4,Small Pepperoni Pizza,11469.73,-104.90,1,11469.73,0.009146,False
5,XS Pepperoni Pizza,11251.28,-96.22,1,11251.28,0.008552,False
6,Medium Multi Topping,18156.68,-150.20,1,18156.68,0.008272,False
7,XL Multi Topping,14890.19,-100.14,1,14890.19,0.006725,False
8,Cheese Bread,10640.79,-68.88,1,10640.79,0.006473,False
9,Wings (6pc),9800.48,-55.96,1,9800.48,0.005710,False


### 2.3.5 Discount Pressure Distribution
- Examine distribution of discount pressure across SKUs.
- Determine whether discounting is concentrated in a few SKUs.
- Prepare inputs for promotion strategy analysis.

In [203]:
discount_pressure_summary = sku_architecture["discount_pressure"].describe()

discount_pressure_summary

count    32.000000
mean      0.006064
std       0.009313
min       0.000000
25%       0.001646
50%       0.003336
75%       0.006536
max       0.049248
Name: discount_pressure, dtype: float64

### 2.3.6 Identify Top Discount Pressure SKUs
- Identify SKUs with the highest discount pressure.
- These SKUs represent candidates for:
  - repricing
  - bundle restructuring
  - promotion targeting.

In [204]:
sku_architecture.sort_values(
    "discount_pressure",
    ascending=False
).head(10)

,SKU,realized_revenue,total_discounts,order_lines,avg_realized_revenue_per_txn,discount_pressure,high_discount_flag
0,XL Garlic Chicken Pizza,12823.79,-631.55,1,12823.79,0.049248,False
1,Lrg Garlic Chicken Pizza,12947.23,-317.31,1,12947.23,0.024508,False
2,XL Combination Pizza,36549.69,-563.53,1,36549.69,0.015418,False
3,XL Hawaiian Delight Pizza,11683.74,-119.94,1,11683.74,0.010266,False
4,Small Pepperoni Pizza,11469.73,-104.90,1,11469.73,0.009146,False
5,XS Pepperoni Pizza,11251.28,-96.22,1,11251.28,0.008552,False
6,Medium Multi Topping,18156.68,-150.20,1,18156.68,0.008272,False
7,XL Multi Topping,14890.19,-100.14,1,14890.19,0.006725,False
8,Cheese Bread,10640.79,-68.88,1,10640.79,0.006473,False
9,Wings (6pc),9800.48,-55.96,1,9800.48,0.005710,False


### 2.4 Modifier & Bundle Behavior Analysis
- Evaluate whether modifiers and bundled items increase realized revenue.
- Identify patterns where add-ons improve ticket size versus where structure may suppress value.
- Focus on revenue-relevant behavior that can inform pricing and promotion decisions.

### 2.4.1 Validate Modifier / Bundle Fields
- Confirm fields needed for modifier and bundle analysis are present.
- This section depends on item naming and realized revenue at the transaction level.
- Fail fast before building behavior tables.

In [205]:
required_cols = {"Name", "SKU", "Category", "realized_revenue"}
missing = required_cols - set(order_items.columns)

if missing:
    raise KeyError(f"order_items missing required columns: {sorted(missing)}")

### 2.4.2 Create Bundle / Modifier Heuristic Flags
- Create simple text-based heuristic flags for bundle-like or modifier-heavy items.
- This is a proxy layer, not a final classification system.
- Useful for identifying where menu structure may influence realized revenue.

In [206]:
name_series = order_items["Name"].fillna("").str.lower()

bundle_terms = [
    "special", "deal", "combo", "family", "meal", "pack"
]

modifier_terms = [
    "add", "extra", "side", "topping", "modifier"
]

order_items["bundle_flag"] = name_series.str.contains(
    "|".join(bundle_terms),
    regex=True
)

order_items["modifier_flag"] = name_series.str.contains(
    "|".join(modifier_terms),
    regex=True
)

order_items[["Name", "bundle_flag", "modifier_flag"]].head(20)

,Name,bundle_flag,modifier_flag
0,Family Special (2 Lrg 2 Toppings),True,True
1,Three Seasons (XL),False,False
2,Large Pizza & Bread Sticks,False,False
3,2 Large Pizza One Topping,False,True
4,Pizza Party Pack (Medium Pizza and 6 Wings),True,False
5,"Superbowl deal (XL, 10 wings, GB)",True,False
6,3 Large Pizzas,False,False
7,Lrg Pepperoni Pizza,False,False
8,XL Pepperoni Pizza,False,False
9,Medium Pepperoni Pizza,False,False


### 2.4.3 Revenue Comparison by Bundle Flag
- Compare realized revenue between bundle-like items and non-bundle items.
- Estimate whether bundle structures are associated with higher ticket capture.
- This is directional and should be interpreted alongside menu context.

In [208]:
bundle_revenue_summary = (
    order_items
    .groupby("bundle_flag", as_index=False)
    .agg(
        order_lines=("Name", "count"),
        realized_revenue=("realized_revenue", "sum")
    )
)

bundle_revenue_summary["avg_realized_revenue_per_txn"] = (
    bundle_revenue_summary["realized_revenue"] /
    bundle_revenue_summary["order_lines"]
)

bundle_revenue_summary

,bundle_flag,order_lines,realized_revenue,avg_realized_revenue_per_txn
0,False,548,971357.06,1772.54938
1,True,19,269477.76,14183.04000


### 2.4.4 Revenue Comparison by Modifier Flag
- Compare realized revenue between modifier-like items and non-modifier items.
- Estimate whether modifier-associated items capture higher realized revenue per transaction.
- This helps evaluate add-on behavior at a directional level.

In [211]:
modifier_revenue_summary = (
    order_items
    .groupby("modifier_flag", as_index=False)
    .agg(
        order_lines=("Name", "count"),
        realized_revenue=("realized_revenue", "sum")
    )
)

modifier_revenue_summary["avg_realized_revenue_per_txn"] = (
    modifier_revenue_summary["realized_revenue"] /
    modifier_revenue_summary["order_lines"]
)

modifier_revenue_summary

,modifier_flag,order_lines,realized_revenue,avg_realized_revenue_per_txn
0,False,535,909855.28,1700.664075
1,True,32,330979.54,10343.110625


### 2.4.5 Top Bundle-Like SKUs by Revenue
- Identify highest-revenue bundle-like SKUs.
- These are the primary candidates for bundle pricing review.
- Focus on bundle structures that materially influence portfolio revenue.

In [212]:
bundle_sku_revenue = (
    order_items[order_items["bundle_flag"]]
    .groupby("SKU", as_index=False)
    .agg(
        transactions=("SKU", "count"),
        realized_revenue=("realized_revenue", "sum")
    )
    .sort_values("realized_revenue", ascending=False)
    .reset_index(drop=True)
)

bundle_sku_revenue.head(15)

,SKU,transactions,realized_revenue
0,Family Special (2 Lrg 2 Toppings),1,228800.60
1,Pizza Party Pack (Medium Pizza and 6 Wings),1,8011.77
2,Buffalo Special,1,7605.34
3,Lrg Half and Half Specialty,1,5313.09
4,XL Half and Half Specialty,1,3588.56
5,Lrg Chef's Special Pizza,1,2472.21
6,Med Half and Half Specialty,1,2192.10
7,XL Chef's Special Pizza,1,2135.65
8,Lrg House Specialty Pizza,1,2020.18
9,Med Chef's Special Pizza,1,1579.23


### 2.4.6 Top Modifier-Like SKUs by Revenue
- Identify highest-revenue modifier-like SKUs.
- These are candidates for add-on optimization or attachment strategy review.
- Focus on items where modifier behavior materially affects realized revenue.

In [213]:
modifier_sku_revenue = (
    order_items[order_items["modifier_flag"]]
    .groupby("SKU", as_index=False)
    .agg(
        transactions=("SKU", "count"),
        realized_revenue=("realized_revenue", "sum")
    )
    .sort_values("realized_revenue", ascending=False)
    .reset_index(drop=True)
)

modifier_sku_revenue.head(15)

,SKU,transactions,realized_revenue
0,Family Special (2 Lrg 2 Toppings),1,228800.60
1,Lrg Multi Topping,1,31150.88
2,2 Large Pizza One Topping,1,21234.93
3,Medium Multi Topping,1,18156.68
4,XL Multi Topping,1,14890.19
5,Small Multi Topping,1,9456.15
6,SL Multi Topping,1,3992.84
7,Square Multi Topping,1,3270.07
8,garlic sauce side,1,5.00
9,jalapeño on the side,2,3.50


### 2.5 Opportunity Prioritization Framework
- Translate diagnostics from Sections 2.1–2.4 into prioritized revenue opportunities.
- Combine revenue concentration, discount pressure, and bundle behavior signals.
- Identify where intervention is most likely to produce measurable impact.
- Opportunity score is a directional metric, not a predictive model

### 2.5.1 Build SKU Opportunity Base Table
- Combine revenue concentration and discount pressure metrics.
- Restrict to revenue-driving SKUs identified earlier.
- Prepare unified table for opportunity scoring.

In [214]:
sku_opportunity = (
    sku_architecture
    .merge(
        sku_revenue[["SKU", "revenue_share", "cumulative_share"]],
        on="SKU",
        how="left"
    )
)

sku_opportunity.head(20)

,SKU,realized_revenue,total_discounts,order_lines,avg_realized_revenue_per_txn,discount_pressure,high_discount_flag,revenue_share,cumulative_share
0,XL Garlic Chicken Pizza,12823.79,-631.55,1,12823.79,0.049248,False,0.010335,0.712900
1,Lrg Garlic Chicken Pizza,12947.23,-317.31,1,12947.23,0.024508,False,0.010434,0.702566
2,XL Combination Pizza,36549.69,-563.53,1,36549.69,0.015418,False,0.029456,0.420088
3,XL Hawaiian Delight Pizza,11683.74,-119.94,1,11683.74,0.010266,False,0.009416,0.741554
4,Small Pepperoni Pizza,11469.73,-104.90,1,11469.73,0.009146,False,0.009244,0.750797
5,XS Pepperoni Pizza,11251.28,-96.22,1,11251.28,0.008552,False,0.009068,0.759865
6,Medium Multi Topping,18156.68,-150.20,1,18156.68,0.008272,False,0.014633,0.653348
7,XL Multi Topping,14890.19,-100.14,1,14890.19,0.006725,False,0.012000,0.692131
8,Cheese Bread,10640.79,-68.88,1,10640.79,0.006473,False,0.008576,0.768440
9,Wings (6pc),9800.48,-55.96,1,9800.48,0.005710,False,0.007898,0.784724


### 2.5.2 Compute Opportunity Score
- Construct simple composite indicator for prioritization.
- Weight factors:
  - revenue_share (importance)
  - discount_pressure (intervention potential)
- Higher scores represent stronger candidates for intervention.

In [215]:
sku_opportunity["opportunity_score"] = (
    sku_opportunity["revenue_share"] *
    sku_opportunity["discount_pressure"]
)

sku_opportunity = sku_opportunity.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

sku_opportunity.head(20)

,SKU,realized_revenue,total_discounts,order_lines,avg_realized_revenue_per_txn,discount_pressure,high_discount_flag,revenue_share,cumulative_share,opportunity_score
0,XL Garlic Chicken Pizza,12823.79,-631.55,1,12823.79,0.049248,False,0.010335,0.712900,0.000509
1,XL Combination Pizza,36549.69,-563.53,1,36549.69,0.015418,False,0.029456,0.420088,0.000454
2,Lrg Garlic Chicken Pizza,12947.23,-317.31,1,12947.23,0.024508,False,0.010434,0.702566,0.000256
3,Lrg Combination Pizza,73447.64,-231.31,1,73447.64,0.003149,False,0.059192,0.307745,0.000186
4,Lrg Pepperoni Pizza,79612.81,-198.88,1,79612.81,0.002498,False,0.064161,0.248553,0.000160
5,XL Pepperoni Pizza,46857.63,-184.15,1,46857.63,0.003930,False,0.037763,0.390632,0.000148
6,Wings(10pc),55991.20,-180.59,1,55991.20,0.003225,False,0.045124,0.352869,0.000146
7,Medium Multi Topping,18156.68,-150.20,1,18156.68,0.008272,False,0.014633,0.653348,0.000121
8,Family Special (2 Lrg 2 Toppings),228800.60,-129.39,1,228800.60,0.000566,False,0.184392,0.184392,0.000104
9,Lrg Multi Topping,31150.88,-127.57,1,31150.88,0.004095,False,0.025105,0.470462,0.000103


### 2.5.3 Identify Tier-1 Intervention Candidates
- Select SKUs with the highest opportunity scores.
- These represent the strongest candidates for pricing or promotion adjustments.
- Focus attention on the top opportunity set.

In [216]:
tier1_candidates = sku_opportunity.head(10)

tier1_candidates

,SKU,realized_revenue,total_discounts,order_lines,avg_realized_revenue_per_txn,discount_pressure,high_discount_flag,revenue_share,cumulative_share,opportunity_score
0,XL Garlic Chicken Pizza,12823.79,-631.55,1,12823.79,0.049248,False,0.010335,0.712900,0.000509
1,XL Combination Pizza,36549.69,-563.53,1,36549.69,0.015418,False,0.029456,0.420088,0.000454
2,Lrg Garlic Chicken Pizza,12947.23,-317.31,1,12947.23,0.024508,False,0.010434,0.702566,0.000256
3,Lrg Combination Pizza,73447.64,-231.31,1,73447.64,0.003149,False,0.059192,0.307745,0.000186
4,Lrg Pepperoni Pizza,79612.81,-198.88,1,79612.81,0.002498,False,0.064161,0.248553,0.000160
5,XL Pepperoni Pizza,46857.63,-184.15,1,46857.63,0.003930,False,0.037763,0.390632,0.000148
6,Wings(10pc),55991.20,-180.59,1,55991.20,0.003225,False,0.045124,0.352869,0.000146
7,Medium Multi Topping,18156.68,-150.20,1,18156.68,0.008272,False,0.014633,0.653348,0.000121
8,Family Special (2 Lrg 2 Toppings),228800.60,-129.39,1,228800.60,0.000566,False,0.184392,0.184392,0.000104
9,Lrg Multi Topping,31150.88,-127.57,1,31150.88,0.004095,False,0.025105,0.470462,0.000103


### 2.5.4 Portfolio Opportunity Summary
- Estimate potential revenue impact if discount pressure were partially reduced.
- This is a directional estimate for prioritization purposes.

In [217]:
potential_recovery = (
    sku_opportunity["total_discounts"].abs().sum()
)

portfolio_opportunity = pd.DataFrame({
    "metric": ["total_discount_pressure", "directional_revenue_recovery"],
    "value": [potential_recovery, potential_recovery]
})

portfolio_opportunity

,metric,value
0,total_discount_pressure,3919.83
1,directional_revenue_recovery,3919.83


### 2.6 Executive Strategy Summary
- Translate analytical findings into actionable revenue strategy.
- Summarize key drivers of revenue concentration, discount impact, and SKU structure.
- Provide clear, prioritized recommendations based on quantified evidence.

### 2.6.1 Key Findings
- Revenue is concentrated within a limited subset of SKUs (Top-80 focus set).
- Discount pressure is present but not structurally required across most SKUs.
- High discount dependence is limited to a small number of SKUs.
- Bundle and modifier behavior contributes to revenue capture but varies by item.

### 2.6.2 Strategic Revenue Levers
- Pricing Discipline:
  - Maintain pricing on core SKUs with low discount pressure.
- Targeted Discounting:
  - Focus discounts on high-pressure SKUs identified in Section 2.3.
- Bundle Optimization:
  - Review high-revenue bundle SKUs for potential repricing.
- Modifier Strategy:
  - Encourage high-value add-ons where modifier behavior increases realized revenue.

### 2.6.3 Priority Action Plan
- Tier 1:
  - Adjust pricing or promotion structure for highest opportunity_score SKUs.
- Tier 2:
  - Optimize bundles with high revenue but unclear pricing efficiency.
- Tier 3:
  - Monitor low-impact SKUs without immediate intervention.

### 2.6.4 Estimated Revenue Opportunity
- Total discount pressure represents a directional estimate of recoverable revenue.
- Based on Section 2.5, discounts are not structurally required across most SKUs.
- Opportunity exists to selectively reduce discounting without materially impacting demand.
- Revenue recovery potential is concentrated within a small subset of SKUs identified as Tier 1 candidates.
- Realized impact depends on disciplined execution of pricing and promotion adjustments.